# Assignment 3 — Code Submission (Jupyter Notebook Template)

**Submit:** this `.ipynb` file inside the required ZIP package, together with the required wav_examples/ folder.

**Group members (Name | ID):**
- `<NAME 1> | <ID 1>`
- `<NAME 2> | <ID 2>`
- `<NAME 3> | <ID 3>`
- `<NAME 4> | <ID 4>`

**Date:** `<DD Month YYYY>`

---

## What the marker should be able to do (end-to-end)
From a fresh environment, the marker should be able to:

1. **Install dependencies** (from your provided dependencies file)
2. **Run training** for each experiment (or run your provided trained checkpoints, if allowed by the task sheet)
3. **Run inference**: input plain text → output a playable `.wav` file
4. **See evidence**: objective metric(s) + qualitative samples (≥10 text prompts with corresponding `.wav` outputs)

---

## Expected artefacts / recommended structure
You can adapt this, but keep it **clear and reproducible**.

- `dependencies.*` (choose one): `requirements.txt` / `environment.yml` / `pyproject.toml`
- `scripts/`
  - `train.py` (or `train_expA.py`, `train_expB.py`, etc.)
  - `synth.py` (single “text → wav” entrypoint recommended)
  - `evaluate.py` (objective metric computation)
- `configs/` (optional but recommended)
  - `expA.yaml`, `expB.yaml`, ...
- `results/`
  - `wavs/` (generated audio samples; **≥10 prompts**)
  - `metrics.json` or `metrics.csv`
  - `spectrograms/` (optional)

> Note: Spectrograms are **optional** unless your task sheet explicitly requires them. If you include them, provide 2–3 representative examples per experiment.

---

## How to use this notebook
- Use the headings below as your “evidence checklist”.
- Add **code cells** directly under each heading for implementation and outputs (tables, logs, plots, audio).
- Keep paths consistent and relative where possible.

---

### Our project at a glance
| | |
|---|---|
| **System** | Matcha-style acoustic model **built from scratch**: Transformer text encoder, Monotonic Alignment Search, duration predictor and a convolutional mel decoder, followed by a **pretrained HiFi-GAN** vocoder (LJSpeech V1) |
| **Experiments** | 2×2 controlled grid: **decoder type** (flow matching vs regression) × **decoder depth** (8 vs 4 residual blocks) → `expA`, `expB`, `expC`, `expD` |
| **Data** | LJSpeech 1.1, 80/10/10 split (seed 42); each experiment trains on the **full training split (10,480 utterances)** |
| **Where to run** | Google Colab, T4 GPU (Runtime → Change runtime type). Run all. LJSpeech and the vocoder download automatically; checkpoints go to `MyDrive/ARTI5001_A3/checkpoints/runs_full/` and resume after a disconnect |
| **Code layout** | Notebook cells write the reusable modules to `tts/` (`%%writefile`), so all code is visible here *and* importable by `scripts/synth.py` |

**Notebook map:**

| Section | Numbered parts | What happens there |
|---|---|---|
| §0 Setup | 0.1 – 0.2 | configuration, paths, seeds, environment report |
| §1.1 Data & preprocessing | 1.1.1 – 1.1.4 (template answers), 1.1.5 – 1.1.12 (code) | integrity checks, raw statistics, split, text and audio preprocessing, statistics after preprocessing |
| §1.2 Pipeline | 1.2.1 – 1.2.4 (template answers), 1.2.5 – 1.2.8 (code) | model modules, vocoder, inference entry point, shape walk-through |
| §1.3 Experiments | 1.3.1 – 1.3.2 (template answers), 1.3.3 – 1.3.6 (code) | configs, training of expA–D, training summary, inference demo |
| §1.4 Evaluation | 1.4.1 – 1.4.5 (template answers), 1.4.1.1 – 1.4.2.3 (code) | test-set metrics with confidence intervals, controlled comparisons, speed, step sweep, validation re-score, ≥10 prompts, spectrograms, attribution |

**How to read the code cells:** every code cell has a numbered heading directly above it with a **"What this cell does"** note (the steps, in order) and the **output to expect**. The first line of each code cell repeats that number, so printed banners can be matched back to their cell.

## 0. Setup
### 0.1 Run configuration, paths and seeds
The cell below is the **only cell a marker should need to edit**. It detects Google Colab versus a local machine and sets every path from that.
- **Colab:** mounts Google Drive and installs the system phonemiser. Checkpoints and results go to Drive so they survive a disconnect. LJSpeech is unpacked to the fast local disk.
- **Local:** runs from the repository root with LJSpeech at `data/LJSpeech-1.1`.

`SMOKE=True` (or the environment variable `A3_SMOKE=1`) runs the whole notebook on a 64-clip subset for 2 epochs. It is an end-to-end check that every cell works, and finishes in minutes.

**What this cell does, step by step:**
1. **Run size:** sets `SEED`, `EPOCHS` and the subset sizes (smaller values in smoke mode).
2. **Environment and paths:** detects Colab or local, and sets every folder the notebook reads or writes.
3. **Dataset:** downloads and extracts LJSpeech unless all 13,100 wavs are already there. A truncated archive or a half-finished extraction (e.g. after a Colab disconnect) is detected and redone.
4. **Folders:** creates the output folders and moves to the project root, so all relative paths work.
5. **Reproducibility:** imports the libraries, seeds Python, NumPy and PyTorch, and picks the fastest device (CUDA → Apple MPS → CPU).
6. **Print helpers:** `banner`, `kv`, `check` and `savefig`, so every later cell prints in the same format.

**Output to expect:** a short table of the run configuration. Check that *LJSpeech folder* says `found` and *Mode* is what you intended.

In [ ]:
# 0.1 Run configuration, paths, seeds
import os, sys, platform, subprocess, shutil, json, time, hashlib, random, re, csv, unicodedata, importlib, importlib.metadata, logging, warnings
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# ---- 1. Run size: the only values that change between a smoke test and the full run ----
SEED = 42
SMOKE = os.environ.get("A3_SMOKE", "0") == "1"        # True: tiny end-to-end check (64 clips, 2 epochs)
EPOCHS = 2 if SMOKE else 300                           # training budget per experiment (identical for all four)
N_TRAIN_SUBSET = 64 if SMOKE else None                 # None = the whole training split (10,480 utterances)
RUN_EXPERIMENTS = ["expA", "expB", "expC", "expD"]     # each teammate keeps only the experiment(s) they train
N_VAL_MONITOR = 32 if SMOKE else 300                   # fixed validation clips scored every epoch
N_TEST_CACHE = 32 if SMOKE else None                   # None = cache features for the whole test split

# ---- 2. Environment detection and paths ----
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run("apt-get install -y -qq espeak-ng > /dev/null", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "phonemizer", "num2words", "numba", "jiwer", "pyworld", "pysptk", "setuptools<81"], check=True)
    PROJECT_ROOT = Path("/content/drive/MyDrive/ARTI5001_A3")             # persists across sessions
    WORK_DIR = Path("/content/work")                                       # fast local disk, wiped on disconnect
    LJ_ARCHIVE = Path("/content/drive/MyDrive/LJ/LJSpeech-1.1.tar.bz2")    # upload the archive here once
    DATA_DIR = WORK_DIR / "LJSpeech-1.1"
else:
    cwd = Path.cwd()
    PROJECT_ROOT = Path(os.environ.get("A3_ROOT", cwd.parent if cwd.name == "notebook" else cwd)).resolve()
    WORK_DIR = PROJECT_ROOT / "data" / "work_full"                         # git-ignored (data/)
    LJ_ARCHIVE = PROJECT_ROOT / "data" / "LJSpeech-1.1.tar.bz2"
    DATA_DIR = Path(os.environ.get("LJSPEECH_DIR", PROJECT_ROOT / "data" / "LJSpeech-1.1"))
    if platform.system() == "Darwin" and Path("/opt/homebrew/lib/libespeak-ng.dylib").exists():
        os.environ.setdefault("PHONEMIZER_ESPEAK_LIBRARY", "/opt/homebrew/lib/libespeak-ng.dylib")

# ---- 3. Get LJSpeech (use the archive if present, otherwise download once; then extract) ----
# Completeness is checked by counting wavs, not by metadata.csv alone: an interrupted extraction leaves
# metadata.csv but only part of wavs/, and an interrupted download leaves a truncated archive.
LJ_N_WAVS, LJ_BYTES = 13100, 2_748_572_632                 # official LJSpeech 1.1 clip count and archive size
def n_wavs():
    return sum(1 for _ in (DATA_DIR / "wavs").glob("*.wav")) if (DATA_DIR / "wavs").exists() else 0
if n_wavs() < LJ_N_WAVS:
    if LJ_ARCHIVE.exists() and LJ_ARCHIVE.stat().st_size != LJ_BYTES:
        print(f"  archive is incomplete ({LJ_ARCHIVE.stat().st_size:,} of {LJ_BYTES:,} bytes): downloading again")
        LJ_ARCHIVE.unlink()
    if not LJ_ARCHIVE.exists():
        LJ_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        part = LJ_ARCHIVE.with_suffix(".part")                 # renamed only once the download is complete
        subprocess.run(["wget", "-q", "-O", str(part), "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"], check=True)
        part.rename(LJ_ARCHIVE)
    archive = LJ_ARCHIVE
    if IN_COLAB:                                               # extracting from local disk is much faster than from Drive
        archive = WORK_DIR / LJ_ARCHIVE.name
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy(LJ_ARCHIVE, archive)
    shutil.rmtree(DATA_DIR, ignore_errors=True)                # start clean after a partial extraction
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    print("  extracting LJSpeech (a few minutes) ...")
    subprocess.run(["tar", "-xjf", str(archive), "-C", str(DATA_DIR.parent)], check=True)
    if archive != LJ_ARCHIVE:
        archive.unlink()
assert n_wavs() == LJ_N_WAVS, f"LJSpeech incomplete: {n_wavs()} of {LJ_N_WAVS} wavs in {DATA_DIR / 'wavs'}"

# ---- 4. Output folders (created if missing) ----
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR     = RESULTS_DIR / "figures"
SPLIT_DIR   = RESULTS_DIR / "splits"
CONFIG_DIR  = PROJECT_ROOT / "configs"
WAV_OUT_DIR = PROJECT_ROOT / "wav_examples"
TTS_DIR     = PROJECT_ROOT / "tts"                     # our package, written by %%writefile cells below
CKPT_ROOT   = PROJECT_ROOT / "checkpoints" / ("smoke" if SMOKE else "runs_full")
VOCODER_PATH = PROJECT_ROOT / "checkpoints" / "hifigan_lj_v1"
PROC_DIR    = WORK_DIR / "processed"                   # phonemes, vocab, mel stats, records
MEL_DIR     = WORK_DIR / "mels"                        # cached log-mels (.pt)
for d in (RESULTS_DIR, FIG_DIR, SPLIT_DIR, CONFIG_DIR, WAV_OUT_DIR, TTS_DIR, PROJECT_ROOT / "scripts", CKPT_ROOT, PROC_DIR, MEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
(TTS_DIR / "__init__.py").touch()

os.chdir(PROJECT_ROOT)                                 # every relative path below is from the project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---- 5. Libraries, seeds, device ----
import numpy as np, pandas as pd, torch, librosa, soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("phonemizer").setLevel(logging.ERROR)
pd.set_option("display.width", 160, "display.max_columns", 20, "display.precision", 3)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# ---- 6. Print helpers used by every cell, so outputs read the same way everywhere ----
def banner(title):
    """Section title between two rules."""
    print(f"\n{'=' * 78}\n{title}\n{'=' * 78}")

def kv(label, value, note=""):
    """One aligned 'label  value  → interpretation' line."""
    print(f"  {label:<38} {value}" + (f"   → {note}" if note else ""))

def check(ok, msg):
    """Prints PASS/FAIL and stops the notebook on FAIL, so a broken step cannot go unnoticed."""
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    assert ok, msg

def savefig(name):
    """Saves the current figure to results/figures/<name>.png for the report."""
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  figure saved → {path.relative_to(PROJECT_ROOT)}")

# ---- 7. Print the configuration ----
banner("0.1 Run configuration")
kv("Environment", "Google Colab" if IN_COLAB else f"local ({platform.system()})")
kv("Mode", "SMOKE TEST (tiny, fast)" if SMOKE else "FULL RUN")
kv("Project root", PROJECT_ROOT)
kv("LJSpeech folder", DATA_DIR, "found" if (DATA_DIR / "metadata.csv").exists() else "MISSING")
kv("Working cache (mels, phonemes)", WORK_DIR)
kv("Checkpoints", CKPT_ROOT)
kv("Device", DEVICE)
kv("Seed", SEED)
kv("Epochs per experiment", EPOCHS)
kv("Training utterances per experiment", N_TRAIN_SUBSET or "all (full training split)")
kv("Experiments trained in this session", ", ".join(RUN_EXPERIMENTS))

### 0.2 Environment report
This records the Python version, key package versions and the hardware. Speed numbers such as RTF only mean something next to the hardware that produced them.

**What this cell does:**
1. Collects the Python version, OS and CPU.
2. Reads the installed version of every key package from its package metadata.
3. Adds the GPU name, memory and CUDA version (if a GPU exists) and the espeak-ng version.
4. Prints everything and saves it to `results/environment.json` for the report.

**Output to expect:** one line per item. No package should say `not installed`.

In [ ]:
# 0.2 Environment report (saved to results/environment.json)
def env_report():
    info = {"python": sys.version.split()[0], "os": platform.platform(), "cpu": platform.processor() or platform.machine()}
    for pkg in ["numpy", "pandas", "torch", "librosa", "soundfile", "phonemizer", "num2words", "numba", "matplotlib"]:
        try:
            info[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            info[pkg] = "not installed"
    info["device"] = str(DEVICE)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info["gpu"] = f"{p.name} ({p.total_memory / 1e9:.1f} GB)"
        info["cuda"] = torch.version.cuda
    try:
        out = subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True).stdout
        info["espeak-ng"] = out.split(":")[1].split()[0]
    except Exception:
        info["espeak-ng"] = "not found"
    return info

ENV = env_report()
banner("0.2 Environment")
for k, v in ENV.items():
    kv(k, v)
(RESULTS_DIR / "environment.json").write_text(json.dumps(ENV, indent=2))
print("\n  saved → results/environment.json")

# 1.1 Dataset handling & pre-processing (Rubric 1.1)

## 1.1.1 Dataset location and structure
- **Dataset:** LJSpeech 1.1 (Ito & Johnson, 2017): 13,100 short clips of one female speaker reading non-fiction books, about 24 hours in total.
- **Local path used:** `data/LJSpeech-1.1/` locally, `/content/work/LJSpeech-1.1/` on Colab (set in §0.1).
- **Files/folders used:**
  - `metadata.csv`: pipe-separated `id | raw transcript | normalised transcript`
  - `wavs/`: `LJxxx-yyyy.wav`, 22,050 Hz mono 16-bit PCM

## 1.1.2 Audio preprocessing (document your choices)
State what you did and why:
- **Sampling rate:** original **22,050 Hz** kept. Every file is verified in §1.1.7. This is also the rate of the pretrained HiFi-GAN vocoder, so no resampling is needed. A resampling fallback exists in `tts/audio.py` for arbitrary input audio.
- **Silence trimming / padding:** leading and trailing silence trimmed with `librosa.effects.trim(top_db=40, frame_length=1024, hop_length=256)`. No padding. The justification and measurements are in §1.1.10.
- **Any filtering / normalization:**
  - No loudness normalisation: levels are already consistent (§1.1.7).
  - 80-band log-mel features use the exact settings of the HiFi-GAN vocoder.
  - The mels are **standardised with a global mean and std computed on the training subset only** (§1.1.10).

## 1.1.3 Text preprocessing (minimum: justify your choice)
- **Normalization:**
  - Starts from the **normalised transcript column** (numbers already spelled out; abbreviations are not).
  - `normalize_text()` then applies, in order:
    1. Unicode → ASCII
    2. curly quotes and dashes standardised
    3. abbreviations expanded
    4. numbers, years, ordinals and currency spelled out
    5. lower-case
    6. parentheses mapped to commas
    7. every symbol except `' , . ? ! ; : -` dropped
    8. whitespace collapsed
  - Justification is in §1.1.9.
- **Tokenizer / phonemizer (if used):**
  - `phonemizer` with the espeak-ng backend, language `en-us`, stress marks kept, punctuation preserved.
  - Each IPA symbol becomes one token.
  - The vocabulary is built from the **training split only**.

## 1.1.4 Train/validation/test split (reproducible)
- **Split ratio:** 80 / 10 / 10 → 10,480 / 1,310 / 1,310 clips.
- **Seed:** 42.
- **How split is created:** sort all IDs, apply `np.random.default_rng(42).permutation`, then cut at 80% and 90%. The ID lists are saved to `results/splits/` with an MD5 fingerprint.
- **Any subset used:**
  - **Training data:** the full training split (10,480 utterances). All four experiments use it. An earlier set of runs on a 3,000-utterance subset (task sheet: 1–3k utterances) is kept for the data-scaling comparison.
  - **Validation monitor:** a fixed 300-clip subset of validation, scored every epoch.
  - **Test:** the full 1,310 clips are kept untouched for final evaluation.

---
### Code in §1.1, in run order
| # | Section | What it does | Main outputs |
|---|---|---|---|
| 1.1.5 | Preprocessing constants | every preprocessing setting in one place | printed settings |
| 1.1.6 | Load & integrity checks | reads `metadata.csv` correctly and checks the dataset is complete | PASS lines |
| 1.1.7 | Raw inspection (*before* preprocessing) | audio and text statistics for all 13,100 clips | stats table, 2 figures |
| 1.1.8 | Split | 80/10/10 split, training subset, validation monitor | `results/splits/`, MD5 |
| 1.1.9 | Text preprocessing | normalisation → phonemes → IDs, vocabulary | `tts/text.py`, `vocab.json`, figure |
| 1.1.10 | Audio preprocessing | trimming, log-mel features, mel normalisation | `tts/audio.py`, mel cache, `mel_stats.json` |
| 1.1.11 | Statistics *after* preprocessing | before vs after comparison | `preprocessing_stats.csv`, figure |
| 1.1.12 | Sanity summary | confirms every input of §1.2–1.3 exists | PASS lines |

The split (1.1.8) comes **before** any statistic that is *fitted*, such as the vocabulary or the mel mean/std, so those can use training data only.

## 1.1.5 Preprocessing constants
**What this cell does:** puts every preprocessing setting in one place:
- the transcript column we read;
- the audio settings: sample rate, trimming threshold, STFT window/hop and mel filterbank;
- the phonemiser settings.

`tts/audio.py` (§1.1.10) uses the same values, and §1.1.10.2 asserts that they match, so the settings cannot silently drift apart. The reasons for each value are in §1.1.9 and §1.1.10.

**Output to expect:** the list of settings.

In [ ]:
# 1.1.5 Preprocessing constants: every choice in one place (justified in §1.1.9 / §1.1.10)
TEXT_COLUMN = "text_norm"          # LJSpeech column 3: numbers/abbreviations already expanded by hand
AUDIO_CFG = dict(
    sr=22050,                      # keep native rate = HiFi-GAN LJ V1 rate
    top_db=40,                     # silence threshold for trimming (dB below the clip's peak)
    n_fft=1024, hop_length=256, win_length=1024,   # 46 ms window, 11.6 ms hop: HiFi-GAN V1 settings
    n_mels=80, fmin=0, fmax=8000,  # HiFi-GAN V1 mel filterbank
)
PHONEMIZER_CFG = dict(language="en-us", backend="espeak", with_stress=True, preserve_punctuation=True)
banner("1.1.5 Preprocessing constants")
kv("Text column", TEXT_COLUMN)
for k, v in AUDIO_CFG.items():
    kv(k, v)
for k, v in PHONEMIZER_CFG.items():
    kv(f"phonemizer.{k}", v)

## 1.1.6 Load metadata and check dataset integrity
`metadata.csv` is read with **`quoting=csv.QUOTE_NONE`**. Some transcripts begin with a double quote, and pandas' default CSV quoting treats that as the start of a quoted field. It then swallows the delimiter, and 16 rows appear to have an *empty* normalised transcript. The cell demonstrates the problem and then runs the integrity checks as assertions, so the notebook stops if the dataset is incomplete.

**What this cell does:**
1. Reads `metadata.csv` twice, once with the default parser and once with `QUOTE_NONE`, and counts the empty transcripts each one produces.
2. Prints one affected line, so the cause is visible.
3. Runs six integrity checks: 13,100 rows; exactly two `|` per line; unique IDs; no empty fields; every row has a wav; every wav has a row.

**Output to expect:** the parser comparison (16 vs 0 empty), six PASS lines, and the first three rows of the table.

In [ ]:
# 1.1.6 Load metadata.csv + integrity checks
META_PATH, WAV_DIR = DATA_DIR / "metadata.csv", DATA_DIR / "wavs"
COLS = ["id", "text_raw", "text_norm"]

raw_lines = META_PATH.read_text(encoding="utf-8").splitlines()        # the file as plain text lines
pipe_counts = Counter(line.count("|") for line in raw_lines)           # must be {2: 13100}
# Same file parsed twice: default CSV quoting (wrong for LJSpeech) and QUOTE_NONE (what we use)
naive = pd.read_csv(META_PATH, sep="|", header=None, names=COLS, dtype=str, keep_default_na=False)
meta = pd.read_csv(META_PATH, sep="|", header=None, names=COLS, dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)
meta["wav_path"] = meta["id"].map(lambda i: str(WAV_DIR / f"{i}.wav"))

banner("1.1.6 Why QUOTE_NONE: default CSV parsing vs QUOTE_NONE")
n_bad = int((naive.text_norm.str.strip() == "").sum())
kv("Empty normalised text, default parser", n_bad, "quote characters swallowed the '|' delimiter")
kv("Empty normalised text, QUOTE_NONE", int((meta.text_norm.str.strip() == "").sum()))
ex = naive[naive.text_norm.str.strip() == ""].id.iloc[0]
print(f"\n  Example {ex}:\n    raw line : {raw_lines[meta.index[meta.id == ex][0]][:110]}...")

banner("1.1.6 Integrity checks")
wav_ids = {p.stem for p in WAV_DIR.glob("*.wav")}
meta_ids = set(meta.id)
check(len(raw_lines) == len(meta) == 13100, f"13,100 metadata rows parsed (got {len(meta):,})")
check(set(pipe_counts) == {2}, f"every line has exactly 2 '|' delimiters {dict(pipe_counts)}")
check(meta.id.is_unique, "no duplicate IDs")
check((meta[COLS].apply(lambda c: c.str.strip() != "")).all().all(), "no empty ID / raw text / normalised text")
check(meta_ids <= wav_ids, f"every metadata row has a wav ({len(meta_ids - wav_ids)} missing)")
check(wav_ids <= meta_ids, f"every wav has a metadata row ({len(wav_ids - meta_ids)} orphan wavs)")
print()
display(meta[["id", "text_raw", "text_norm"]].head(3))

## 1.1.7 Inspect the raw data, before any preprocessing
Every one of the 13,100 clips is analysed:
- **File format:** sample rate, channels, sample format.
- **Duration.**
- **Levels:** peak level, and the RMS of the speech region.
- **Silence:** how much leading and trailing silence each clip has. This is measured with the same rule used later for trimming, but nothing is modified yet.

For the text, we count characters, words and speaking rate, and measure what LJSpeech's normalised column already fixes compared with the raw column. These measurements are the evidence behind each preprocessing decision in §1.1.9 and §1.1.10. The results are cached to `processed/raw_audio_stats.csv`, so a re-run is instant.

### 1.1.7.1 Raw statistics: audio and text
**What this cell does:**
1. `analyse_raw_clip()` opens one wav and measures:
   - its format (sample rate, channels, sample type) and duration;
   - the peak level and the RMS level of the speech region, in dBFS;
   - the leading and trailing silence, using the same `librosa.effects.trim` rule as the real trimming later. Nothing is modified yet.
2. Runs it on all 13,100 wavs with 8 threads, or loads the cached CSV.
3. Adds text statistics: characters, words and characters per second, plus counts of digits, abbreviations and non-ASCII characters in the raw and normalised columns.

**Output to expect:**
- the format count (one format for all files);
- a statistics table with mean, std, min, median and max;
- total hours, total edge silence, level spread and clipped clips;
- the raw vs normalised column comparison, with two examples.

In [ ]:
# 1.1.7.1 Raw audio + text inspection (all 13,100 clips)
def analyse_raw_clip(path):
    """One wav -> format, duration, levels and edge silence (measured only, nothing is changed)."""
    info = sf.info(path)
    y, sr = sf.read(path, dtype="float32")
    # start/end = first and last sample louder than top_db below the peak; everything outside is "silence"
    _, (start, end) = librosa.effects.trim(y, top_db=AUDIO_CFG["top_db"], frame_length=AUDIO_CFG["win_length"],
                                           hop_length=AUDIO_CFG["hop_length"])
    speech = y[start:end]                                    # RMS is measured on speech only, so silence does not lower it
    return dict(sr=sr, channels=info.channels, subtype=info.subtype, dur_s=len(y) / sr,
                peak_dbfs=20 * np.log10(np.abs(y).max() + 1e-9),
                rms_dbfs=20 * np.log10(np.sqrt(np.mean(speech ** 2)) + 1e-9),
                lead_sil_s=start / sr, trail_sil_s=(len(y) - end) / sr, trimmed_dur_s=(end - start) / sr)

raw_stats_path = PROC_DIR / "raw_audio_stats.csv"
if raw_stats_path.exists():                                  # cached from an earlier run
    audio_stats = pd.read_csv(raw_stats_path)
else:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=8) as ex:
        audio_stats = pd.DataFrame(list(ex.map(analyse_raw_clip, meta.wav_path)))
    audio_stats.insert(0, "id", meta.id.values)
    audio_stats.to_csv(raw_stats_path, index=False)
    print(f"  analysed {len(audio_stats):,} wavs in {time.perf_counter() - t0:.0f}s")
meta = meta.merge(audio_stats, on="id")

# Text statistics on the raw and normalised columns
meta["n_chars"] = meta.text_norm.str.len()
meta["n_words"] = meta.text_norm.str.split().str.len()
meta["chars_per_s"] = meta.n_chars / meta.dur_s
ABBREV_RE = r"\b(?:Mr|Mrs|Dr|St|Co|Jr|Gen|Capt|Col|Lt|Sgt|Rev|Hon|Esq|Ltd|Maj|Ft)\."   # common titles/abbreviations
# How much does LJSpeech's own normalised column already fix? (decides what normalize_text must still do)
digits_raw, digits_norm = meta.text_raw.str.contains(r"\d").sum(), meta.text_norm.str.contains(r"\d").sum()
abbr_raw, abbr_norm = meta.text_raw.str.contains(ABBREV_RE).sum(), meta.text_norm.str.contains(ABBREV_RE).sum()
non_ascii = Counter(ch for t in meta.text_norm for ch in t if ord(ch) > 127)

banner("1.1.7.1 Audio format (all clips)")
fmt = meta.groupby(["sr", "channels", "subtype"]).size()
for (sr_, ch_, st_), n in fmt.items():
    kv(f"{sr_} Hz / {ch_} ch / {st_}", f"{n:,} files")
print(f"  → {'all files share one format: no resampling or channel mixing needed' if len(fmt) == 1 else 'MIXED formats: resampling needed'}")

banner("1.1.7.1 Raw statistics (before preprocessing)")
raw_table = meta[["dur_s", "n_chars", "n_words", "chars_per_s", "peak_dbfs", "rms_dbfs", "lead_sil_s", "trail_sil_s"]].describe().T
raw_table = raw_table[["mean", "std", "min", "50%", "max"]].rename(columns={"50%": "median"})
display(raw_table.round(2))
kv("Total audio", f"{meta.dur_s.sum() / 3600:.2f} h")
kv("Total silence at clip edges", f"{(meta.lead_sil_s + meta.trail_sil_s).sum() / 3600:.2f} h",
   f"{(meta.lead_sil_s + meta.trail_sil_s).sum() / meta.dur_s.sum():.1%} of all audio")
kv("RMS level spread (std)", f"{meta.rms_dbfs.std():.2f} dB", "small spread: recordings are already level-matched")
kv("Clips that clip (peak ≥ -0.1 dBFS)", int((meta.peak_dbfs >= -0.1).sum()))

banner("1.1.7.1 Raw vs normalised transcript column")
kv("Transcripts that differ", f"{(meta.text_raw != meta.text_norm).sum():,}")
kv("Contain digits (raw → normalised)", f"{digits_raw:,} → {digits_norm:,}")
kv("Contain abbreviations (raw → normalised)", f"{abbr_raw:,} → {abbr_norm:,}")
kv("Non-ASCII characters in normalised text", dict(non_ascii.most_common(8)))
for _, r in meta[meta.text_raw.str.contains(r"\d")].head(2).iterrows():
    print(f"\n    RAW : {r.text_raw[:100]}\n    NORM: {r.text_norm[:100]}")

### 1.1.7.2 Plots: what the dataset contains
**What this cell does:** draws two figures from the statistics above, saves them to `results/figures/`, and plays the first clip.
- **Dataset overview** (`1_1_B_raw_overview.png`): clip duration, transcript length, words per clip, characters vs duration, speaking rate, and the 20 most frequent words.
- **Levels and silence** (`1_1_B_raw_levels_silence.png`): peak level, speech RMS level, leading silence and trailing silence.

**How to read it:**
- A tight, nearly linear characters-vs-duration cloud means one steady reader. Durations are short enough for full-clip training batches.
- The narrow RMS histogram supports *no loudness normalisation*.
- The silence histograms support the trimming decision in §1.1.10: leading silence is near zero, trailing silence is not.

In [ ]:
# 1.1.7.2 Plots: what the dataset contains
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax[0, 0].hist(meta.dur_s, bins=50, color="tab:blue"); ax[0, 0].set(title="Clip duration", xlabel="seconds", ylabel="clips")
ax[0, 1].hist(meta.n_chars, bins=50, color="tab:orange"); ax[0, 1].set(title="Transcript length", xlabel="characters", ylabel="clips")
ax[0, 2].hist(meta.n_words, bins=range(0, 42), color="tab:green"); ax[0, 2].set(title="Words per clip", xlabel="words", ylabel="clips")
ax[1, 0].scatter(meta.n_chars, meta.dur_s, s=1, alpha=0.2); ax[1, 0].set(title="Characters vs duration", xlabel="characters", ylabel="seconds")
ax[1, 1].hist(meta.chars_per_s, bins=50, color="tab:purple"); ax[1, 1].set(title="Speaking rate", xlabel="characters / second", ylabel="clips")
words = Counter(w for t in meta.text_norm.str.lower() for w in re.findall(r"[a-z']+", t))
top = words.most_common(20)[::-1]
ax[1, 2].barh([w for w, _ in top], [n for _, n in top], color="tab:gray"); ax[1, 2].set(title="20 most frequent words", xlabel="count")
fig.suptitle("LJSpeech before preprocessing: duration, text length, speaking rate, vocabulary", fontsize=13)
plt.tight_layout(); savefig("1_1_B_raw_overview"); plt.show()

fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
ax[0].hist(meta.peak_dbfs, bins=50, color="tab:red"); ax[0].set(title="Peak level", xlabel="dBFS", ylabel="clips")
ax[1].hist(meta.rms_dbfs, bins=50, color="tab:red"); ax[1].set(title="Speech RMS level", xlabel="dBFS")
ax[2].hist(meta.lead_sil_s, bins=50, color="tab:cyan"); ax[2].set(title="Leading silence", xlabel="seconds")
ax[3].hist(meta.trail_sil_s, bins=50, color="tab:cyan"); ax[3].set(title="Trailing silence", xlabel="seconds")
fig.suptitle(f"Raw audio levels and edge silence (silence = below {AUDIO_CFG['top_db']} dB from peak)", fontsize=12)
plt.tight_layout(); savefig("1_1_B_raw_levels_silence"); plt.show()

ex_id = meta.id.iloc[0]
y_ex, _ = sf.read(meta.wav_path.iloc[0], dtype="float32")
print(f"Example clip {ex_id}: \"{meta.text_norm.iloc[0]}\"")
display(Audio(y_ex, rate=AUDIO_CFG["sr"]))

## 1.1.8 Train / validation / test split (template 1.1.4)
The split is made **before** any statistic is fitted: the phoneme vocabulary and the mel mean and std come from training data only, so nothing about the validation or test sets leaks into the model.
- **Ratio:** 80/10/10. Two team members had already used it, and 1,310 test clips is a large enough test set.
- **Subsets:** each is the prefix of a seeded permutation, so it is deterministic. Here the training "subset" is the whole training split (`N_TRAIN_SUBSET = None`); the same code drew the 3,000-utterance subset in our earlier runs.

**What this cell does:**
1. `make_split()` sorts all IDs, shuffles them with seed 42 and cuts at 80% and 90%.
2. `seeded_subset()` draws the training subset (from train), the validation monitor (from val) and the cached test set (from test).
3. Writes every ID list to `results/splits/*.txt`, plus an MD5 fingerprint of the train/val/test lists. Anyone can re-create the split and compare the fingerprint.
4. Checks that train, val and test do not overlap and that each subset comes from its own split.
5. Compares the mean duration, transcript length, speaking rate and level per split.

**Output to expect:** split sizes, the MD5, five PASS lines, and a table of near-identical means per split.

In [ ]:
# 1.1.8 Reproducible split + training subset + validation monitor
def make_split(ids, ratio=(0.8, 0.1, 0.1), seed=SEED):
    """Sort first so the result does not depend on file order, then one seeded shuffle and two cuts."""
    ids = np.random.default_rng(seed).permutation(sorted(ids))
    n_tr, n_va = int(ratio[0] * len(ids)), int(ratio[1] * len(ids))
    return {"train": sorted(ids[:n_tr]), "val": sorted(ids[n_tr:n_tr + n_va]), "test": sorted(ids[n_tr + n_va:])}

def seeded_subset(ids, n, seed=SEED):
    """First n IDs of a seeded shuffle: a deterministic random sample."""
    return sorted(np.random.default_rng(seed).permutation(sorted(ids))[:n])

SPLIT = make_split(meta.id)
SUBSETS = {"train_subset": seeded_subset(SPLIT["train"], N_TRAIN_SUBSET),
           "val_monitor": seeded_subset(SPLIT["val"], N_VAL_MONITOR),
           "test_cache": SPLIT["test"] if N_TEST_CACHE is None else seeded_subset(SPLIT["test"], N_TEST_CACHE)}
for name, ids in {**SPLIT, **SUBSETS}.items():
    (SPLIT_DIR / f"{name}.txt").write_text("\n".join(ids) + "\n")
# Fingerprint of the exact ID lists: same MD5 = same split, on any machine
SPLIT_MD5 = hashlib.md5("|".join(",".join(SPLIT[k]) for k in ("train", "val", "test")).encode()).hexdigest()
(SPLIT_DIR / "split_md5.txt").write_text(SPLIT_MD5 + "\n")
meta["split"] = meta.id.map({i: k for k, ids in SPLIT.items() for i in ids})

banner("1.1.8 Split (ratio 80/10/10, seed 42)")
for k in ("train", "val", "test"):
    kv(k, f"{len(SPLIT[k]):,} clips", f"{len(SPLIT[k]) / len(meta):.0%}")
kv("Split fingerprint (MD5)", SPLIT_MD5, "identical for every experiment")
check(not (set(SPLIT["train"]) & set(SPLIT["val"])), "train ∩ val = ∅")
check(not (set(SPLIT["train"]) & set(SPLIT["test"])), "train ∩ test = ∅")
check(not (set(SPLIT["val"]) & set(SPLIT["test"])), "val ∩ test = ∅")
check(set(SUBSETS["train_subset"]) <= set(SPLIT["train"]), f"training subset ({len(SUBSETS['train_subset']):,}) drawn only from train")
check(set(SUBSETS["val_monitor"]) <= set(SPLIT["val"]), f"validation monitor ({len(SUBSETS['val_monitor'])}) drawn only from val")

print("\n  Are the splits representative? (means per split)")
rep = meta.assign(group=meta.split).groupby("group")[["dur_s", "n_chars", "chars_per_s", "rms_dbfs"]].mean()
rep.loc["train_subset"] = meta[meta.id.isin(SUBSETS["train_subset"])][["dur_s", "n_chars", "chars_per_s", "rms_dbfs"]].mean()
rep["hours"] = [meta[meta.split == g].dur_s.sum() / 3600 if g in SPLIT else meta[meta.id.isin(SUBSETS[g])].dur_s.sum() / 3600 for g in rep.index]
display(rep.round(2))
print("  → all splits have near-identical duration, length, speaking-rate and level distributions")

## 1.1.9 Text preprocessing: normalisation → phonemes → IDs
| Step | Choice | Why |
|---|---|---|
| Source column | **normalised column** (`text_norm`) | §1.1.7: 1,489 raw transcripts contain digits and the normalised column has **none**: numbers, years and money are spelled out by hand ("1455" → "fourteen fifty-five"), which is more accurate than rules. |
| Unicode → ASCII, standardise quotes and dashes | `NFKD` + an ASCII map | A few non-ASCII letters (e.g. "é") and curly quotes would otherwise become rare, separate symbols with too few training examples. |
| Abbreviations | expand (`Mr.` → `mister` …), list from keithito/tacotron | §1.1.7: the normalised column **still contains abbreviations** (573 clips, e.g. "Mr.", "St."). LJSpeech only expanded numbers. Without expansion the "." would also be read as a sentence end (a pause) in the middle of a name. |
| Numbers, years, ordinals, times, currency | spell out with `num2words` (`$3.50` → "three dollars, fifty cents"; `1963` → "nineteen sixty-three"; `10:30` → "ten thirty") | The training text has no digits left, so **arbitrary input text** at inference must be brought to the same form; the model has never heard a digit token. |
| Lower-case | yes | Case carries no pronunciation information for English TTS, and it halves the symbol set before phonemisation. |
| Punctuation | keep `, . ? ! ; :` (and `'` `-` inside words); parentheses → comma; drop quotes and brackets | Commas and full stops mark **pauses**, and question marks change **intonation**. Quotes have no sound. |
| Tokens | **IPA phonemes** from espeak-ng (`en-us`, with stress marks), one token per symbol | English spelling is ambiguous ("read", "though", "colonel"). Phonemes give the model the pronunciation directly, so it learns *how to say* sounds rather than spelling rules. Stress marks help prosody. |
| Vocabulary | built from the **training split** only, plus `<pad>`=0 | No leakage. Unknown symbols in val/test are counted below. |

This section has three cells: **1.1.9.1** writes the text module, **1.1.9.2** tests the normalisation, **1.1.9.3** phonemises the dataset and builds the vocabulary.

### 1.1.9.1 Text front-end module (`tts/text.py`)
**What this cell does:** writes the module to disk with `%%writefile`, so training, inference and `scripts/synth.py` all import exactly the same code. It defines:
- `normalize_text()`: the normalisation steps from the table above, in order.
- `expand_numbers()`: spells out money, then times, ordinals, years, decimals and finally plain integers. The order matters: `$3.50` must be read as money before the decimal rule sees `3.50`, and `1963` as a year before the integer rule sees it.
- `to_phonemes()`: normalised text → IPA string with espeak-ng.
- `build_vocab()`, `phonemes_to_ids()`: IPA symbol ↔ integer ID.
- `text_to_ids()`: the whole chain in one call, used at inference.

**Output to expect:** `Writing tts/text.py` (or `Overwriting` on a re-run).

In [ ]:
%%writefile tts/text.py
"""Text front-end: normalize_text() -> IPA phonemes (espeak-ng via phonemizer) -> integer ids.
Abbreviation list adapted from keithito/tacotron (https://github.com/keithito/tacotron/blob/master/text/cleaners.py).
"""
import json, logging, re, unicodedata
from num2words import num2words

PAD = "<pad>"
PAD_ID = 0                                                                              # padding id, ignored by the model

# (pattern, spoken form); "mrs" is listed before "mr" so the longer abbreviation wins
_ABBREVIATIONS = [(re.compile(rf"\b{abbr}\.", re.IGNORECASE), full) for abbr, full in [
    ("mrs", "misess"), ("mr", "mister"), ("dr", "doctor"), ("st", "saint"), ("co", "company"), ("jr", "junior"),
    ("maj", "major"), ("gen", "general"), ("drs", "doctors"), ("rev", "reverend"), ("lt", "lieutenant"),
    ("hon", "honorable"), ("sgt", "sergeant"), ("capt", "captain"), ("esq", "esquire"), ("ltd", "limited"),
    ("col", "colonel"), ("ft", "fort")]]
_PUNCT_MAP = {"“": '"', "”": '"', "‘": "'", "’": "'", "—": ", ", "–": ", ", "--": ", "}   # dashes read as a pause


def _money(amount, unit, sub_unit):
    """'3.50', 'dollar', 'cent' -> 'three dollars, fifty cents'."""
    whole, _, frac = amount.partition(".")
    whole, frac = int(whole or 0), int((frac + "00")[:2]) if frac else 0
    parts = []
    if whole:
        parts.append(f"{num2words(whole)} {unit}{'s' if whole != 1 else ''}")
    if frac:
        parts.append(f"{num2words(frac)} {sub_unit}{'s' if frac != 1 else ''}")
    return ", ".join(parts) or f"zero {unit}s"


def expand_numbers(text):
    """Digits -> words. Most specific patterns first (money, times, ordinals, years), plain integers last."""
    text = re.sub(r"(\d),(\d{3})", r"\1\2", text)                                      # 1,000 -> 1000
    text = re.sub(r"\$(\d+(?:\.\d+)?)", lambda m: _money(m.group(1), "dollar", "cent"), text)
    text = re.sub(r"£(\d+(?:\.\d+)?)", lambda m: _money(m.group(1), "pound", "penny"), text)
    text = re.sub(r"\b(\d{1,2}):(\d{2})\b", lambda m: num2words(int(m.group(1))) + (                    # times
        " o'clock" if m.group(2) == "00" else " " + ("oh " if m.group(2)[0] == "0" else "") + num2words(int(m.group(2)))), text)
    text = re.sub(r"\b(\d+)(st|nd|rd|th)\b", lambda m: num2words(int(m.group(1)), to="ordinal"), text)
    text = re.sub(r"\b(1[1-9]\d\d|20\d\d)\b", lambda m: num2words(int(m.group(1)), to="year"), text)  # years
    text = re.sub(r"\b\d+\.\d+\b", lambda m: num2words(float(m.group(0))), text)
    text = re.sub(r"\b\d+\b", lambda m: num2words(int(m.group(0))), text)
    return text


def normalize_text(text):
    """Make arbitrary English text look like LJSpeech's normalised transcripts (see notebook §1.1.9)."""
    for src, dst in _PUNCT_MAP.items():
        text = text.replace(src, dst)
    for pattern, full in _ABBREVIATIONS:
        text = pattern.sub(full, text)
    text = expand_numbers(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()     # é -> e
    text = text.lower()
    text = re.sub(r"[()\[\]]", ", ", text)                                             # parentheses -> pause
    text = re.sub(r"[^a-z' ,.?!;:-]", " ", text)                                       # drop quotes, symbols
    text = re.sub(r"\s+([,.?!;:])", r"\1", text)                                       # no space before punctuation
    text = re.sub(r"([,;:])(?:\s*[,;:])+", r"\1", text)                                # ", ," -> ","
    text = re.sub(r"[,;:]+([.?!])", r"\1", text)                                        # ",." -> "."
    text = re.sub(r"([,;:.?!])(?=[a-z])", r"\1 ", text)                                 # space after punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text.lstrip(",;: ")                                                         # trailing commas kept: prosody


_BACKEND = None


def to_phonemes(texts, njobs=1):
    """List of normalised strings -> list of IPA strings (espeak-ng en-us, stress marks, punctuation kept)."""
    global _BACKEND
    if _BACKEND is None:
        from phonemizer.backend import EspeakBackend
        quiet = logging.getLogger("phonemizer"); quiet.setLevel(logging.ERROR)
        _BACKEND = EspeakBackend("en-us", preserve_punctuation=True, with_stress=True, logger=quiet)
    return _BACKEND.phonemize(list(texts), strip=True, njobs=njobs)


def build_vocab(phoneme_strings):
    """One id per distinct character in the IPA strings (sorted, so ids are stable); 0 is reserved for padding."""
    symbols = sorted(set("".join(phoneme_strings)))
    return {PAD: PAD_ID, **{s: i + 1 for i, s in enumerate(symbols)}}


def phonemes_to_ids(phonemes, vocab):
    """Unknown symbols are dropped (none occur in val/test, see notebook §1.1.9.3)."""
    return [vocab[ch] for ch in phonemes if ch in vocab]


def text_to_ids(text, vocab):
    """Arbitrary text -> ids (the full front-end used at inference)."""
    return phonemes_to_ids(to_phonemes([normalize_text(text)])[0], vocab)


def save_vocab(vocab, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=0)


def load_vocab(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

### 1.1.9.2 Normalisation test
**What this cell does:**
1. Runs `normalize_text()` on four deliberately hard sentences. Between them they contain abbreviations, money, a date, a time, ordinals, curly quotes, an accented letter and parentheses, so every rule can be checked by eye.
2. Applies it to all 13,100 LJSpeech transcripts (stored as `meta["text"]`, the text used from here on) and counts how many change beyond lower-casing.

**Output to expect:** IN/OUT pairs for the four sentences, the number of changed transcripts, and three examples.

In [ ]:
# 1.1.9.2 Normalisation on hard inputs: what arbitrary user text looks like after each step
import tts.text as T; importlib.reload(T)

stress_tests = [
    "Mr. Smith paid $3.50 for 2 books on 12 Jan 1963.",
    "Dr. Watson arrived at 221st St. — just after 10:30!",
    "The café's “special” costs £4 (tax included).",
    "Is it the 1st or the 21st of May?",
]
banner("1.1.9.2 Text normalisation on stress-test sentences")
for s in stress_tests:
    print(f"  IN : {s}\n  OUT: {T.normalize_text(s)}\n")

# Apply to LJSpeech's normalised column: mostly lower-casing + quote removal, a safety net for leftovers
meta["text"] = meta[TEXT_COLUMN].map(T.normalize_text)
changed = meta[meta.text != meta[TEXT_COLUMN].str.lower()]
kv("LJSpeech transcripts changed beyond lower-casing", f"{len(changed):,} / {len(meta):,}", "abbreviations, quotes, parentheses, non-ASCII")
for _, r in changed.head(3).iterrows():
    print(f"    {r.id}: {r[TEXT_COLUMN][:90]}\n    {' ' * len(r.id)}  {r.text[:90]}")

### 1.1.9.3 Phonemes and vocabulary
**What this cell does:**
1. Phonemises all 13,100 normalised transcripts once with espeak-ng (4 parallel jobs) and caches them to `processed/phonemes.json`. This takes a few minutes the first time and is instant afterwards.
2. Builds the symbol vocabulary from the **training split only** and saves it to `vocab.json`.
3. Checks whether val or test contain symbols never seen in training.
4. Shows the full front-end (text → IPA → IDs) for three clips.
5. Plots how often each symbol occurs and how long the phoneme sequences are.

**Output to expect:** the vocabulary size, the unseen-symbol counts (should be none), an example table, and `1_1_D_phonemes.png`.

In [ ]:
# 1.1.9.3 Phonemise every transcript once (cached), build the vocabulary from TRAIN only
ph_path = PROC_DIR / "phonemes.json"
if ph_path.exists() and set(json.loads(ph_path.read_text())) == set(meta.id):          # reuse the cache only if complete
    PHONEMES = json.loads(ph_path.read_text())
else:
    t0 = time.perf_counter()
    PHONEMES = dict(zip(meta.id, T.to_phonemes(meta.text.tolist(), njobs=4)))
    ph_path.write_text(json.dumps(PHONEMES, ensure_ascii=False))
    print(f"  phonemised {len(PHONEMES):,} transcripts in {time.perf_counter() - t0:.0f}s")
meta["phonemes"] = meta.id.map(PHONEMES)

VOCAB = T.build_vocab(meta[meta.split == "train"].phonemes)                          # training split only: no leakage
T.save_vocab(VOCAB, PROC_DIR / "vocab.json"); T.save_vocab(VOCAB, RESULTS_DIR / "vocab.json")
meta["n_phonemes"] = meta.phonemes.map(lambda p: len(T.phonemes_to_ids(p, VOCAB)))

banner("1.1.9.3 Phoneme vocabulary (built from the training split only)")
kv("Vocabulary size", f"{len(VOCAB)} symbols", "incl. <pad>=0, IPA phones, stress marks ˈ ˌ, length mark ː, punctuation, space")
for split in ("val", "test"):
    unseen = Counter(ch for p in meta[meta.split == split].phonemes for ch in p if ch not in VOCAB)
    kv(f"Symbols in {split} never seen in train", dict(unseen) or "none", "no out-of-vocabulary tokens" if not unseen else "dropped at tokenisation")
kv("Phonemes per clip (mean ± std)", f"{meta.n_phonemes.mean():.1f} ± {meta.n_phonemes.std():.1f}")

print("\n  The full front-end on three clips:")
demo = meta.head(3).assign(ids=lambda d: d.phonemes.map(lambda p: T.phonemes_to_ids(p, VOCAB)))
display(pd.DataFrame({"id": demo.id, "normalised text": demo.text.str[:45] + "…",
                      "phonemes (IPA)": demo.phonemes.str[:45] + "…",
                      "ids (first 12)": demo.ids.map(lambda x: str(x[:12])[:-1] + ", …]"),
                      "#ids": demo.ids.map(len)}).set_index("id"))

freq = Counter(ch for p in meta[meta.split == "train"].phonemes for ch in p)
fig, ax = plt.subplots(1, 2, figsize=(17, 3.8), gridspec_kw={"width_ratios": [3, 1]})
syms, counts = zip(*freq.most_common())
ax[0].bar(range(len(syms)), counts, color="tab:blue"); ax[0].set_xticks(range(len(syms)), [repr(s)[1:-1] for s in syms], fontsize=8)
ax[0].set(title="Phoneme symbol frequency (training split)", ylabel="count (log)", yscale="log")
ax[1].hist(meta.n_phonemes, bins=50, color="tab:orange"); ax[1].set(title="Phoneme sequence length", xlabel="tokens per clip", ylabel="clips")
plt.tight_layout(); savefig("1_1_D_phonemes"); plt.show()

## 1.1.10 Audio preprocessing: sample rate, trimming, mel features, mel normalisation
| Step | Choice | Why (evidence) |
|---|---|---|
| Sample rate | **keep 22,050 Hz**; resample only if a file differs | All 13,100 files are already 22,050 Hz mono (§1.1.7), and the pretrained HiFi-GAN vocoder was trained at 22,050 Hz. Resampling would add cost and could only lose information. |
| Silence trimming | **trim leading and trailing silence**, `top_db=40` relative to the clip peak, 1024/256 frames | §1.1.7: LJSpeech clips start almost immediately (leading silence ≈ 0 s) but end with **≈ 80 ms of trailing silence** in nearly every clip. Silence carries no text, yet Monotonic Alignment Search must assign every frame to some phoneme, so these frames would be attached to the **last phoneme** (usually the final punctuation), inflating its duration and teaching the model a trailing pause it cannot predict from text. Trimming makes the clip edges consistent. The cost is small (≈ 1% of the audio), and `top_db=40` is well below the speech level (speech RMS ≈ 19 dB below peak), so word endings are kept; the over-trimming guard below checks this. |
| Loudness | **no normalisation** | Speech RMS has a small spread across clips (§1.1.7) and no clip is clipped, so normalising would change almost nothing. |
| Features | 80-band **log-mel**: n_fft 1024, hop 256 (11.6 ms), win 1024, fmin 0, **fmax 8 kHz**, log with floor 1e-5 | These are **exactly the settings the pretrained HiFi-GAN V1 was trained on**. The vocoder only produces clean audio from mels that look like its training mels. |
| Mel normalisation | standardise every mel with **one global mean and std, computed on the training subset only** | Unnormalised log-mels sit around −5.5 with std ≈ 2. The flow-matching decoder starts from N(0,1) noise, and targets on the same scale make training stable; the regression decoder benefits the same way. Using training statistics only avoids leakage. At inference the mel is de-normalised before the vocoder. |

This section has four cells: **1.1.10.1** writes the audio module, **1.1.10.2** checks trimming, **1.1.10.3** caches the features used for training, **1.1.10.4** computes the mel normalisation statistics.

### 1.1.10.1 Audio front-end module (`tts/audio.py`)
**What this cell does:** writes the module to disk. It defines two functions:
- `load_wav()`: reads a file, mixes to mono, resamples to 22,050 Hz if needed, and trims leading/trailing silence.
- `mel_spectrogram()`: the HiFi-GAN-compatible log-mel. It reflect-pads the signal, takes the STFT magnitude, applies the 80-band mel filterbank, and takes the log with a floor of 1e-5, so silence does not become −∞.

**Output to expect:** `Writing tts/audio.py`.

In [ ]:
%%writefile tts/audio.py
"""Audio front-end: load -> (resample) -> trim silence -> 80-band log-mel.
Mel extraction adapted from HiFi-GAN (Kong et al., 2020): https://github.com/jik876/hifi-gan/blob/master/meldataset.py
so features match the pretrained HiFi-GAN LJ V1 vocoder exactly.
"""
import librosa, numpy as np, soundfile as sf, torch

SR, N_FFT, HOP, WIN = 22050, 1024, 256, 1024
N_MELS, FMIN, FMAX = 80, 0, 8000
TOP_DB = 40

# Built once at import: (80, 513) mel filterbank and the STFT window
MEL_BASIS = torch.from_numpy(librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)).float()
WINDOW = torch.hann_window(WIN)


def load_wav(path, trim=True):
    """Returns float32 mono audio in [-1, 1] at SR Hz, leading/trailing silence removed if trim=True."""
    wav, sr = sf.read(path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    if trim:
        wav, _ = librosa.effects.trim(wav, top_db=TOP_DB, frame_length=WIN, hop_length=HOP)
    return wav


def mel_spectrogram(wav):
    """wav: 1-D numpy/tensor -> log-mel tensor (80, frames)."""
    wav = torch.as_tensor(wav, dtype=torch.float32)
    pad = (N_FFT - HOP) // 2                                                  # HiFi-GAN padding: frames = samples / HOP
    wav = torch.nn.functional.pad(wav[None, None], (pad, pad), mode="reflect")[0]
    spec = torch.stft(wav, N_FFT, hop_length=HOP, win_length=WIN, window=WINDOW, center=False, return_complex=True)
    spec = torch.sqrt(spec.real ** 2 + spec.imag ** 2 + 1e-9)                 # magnitude spectrogram (1, 513, frames)
    return torch.log(torch.clamp(MEL_BASIS @ spec, min=1e-5))[0]              # mel filterbank, then log with a floor

### 1.1.10.2 Trimming check
**What this cell does:**
1. Asserts that the constants in `tts/audio.py` equal the settings in §1.1.5.
2. Picks the clip with the **most** edge silence (the worst case) and plots its waveform, with the removed parts shaded, and its log-mel before and after trimming.
3. Measures what trimming removes over all 13,100 clips. This is the **over-trimming guard**: the cell fails if any clip loses more than half its length.
4. Plays that clip before and after trimming, so you can hear that no speech is cut.

**Output to expect:** `1_1_E_trim_example.png`, the trimming statistics, a PASS line and two audio players.

In [ ]:
# 1.1.10.2 Trimming: before/after on one clip + an over-trimming guard over the whole dataset
import tts.audio as A; importlib.reload(A)
assert (A.SR, A.N_FFT, A.HOP, A.N_MELS, A.FMAX, A.TOP_DB) == (AUDIO_CFG["sr"], AUDIO_CFG["n_fft"], AUDIO_CFG["hop_length"],
                                                              AUDIO_CFG["n_mels"], AUDIO_CFG["fmax"], AUDIO_CFG["top_db"])
ex = meta.loc[(meta.lead_sil_s + meta.trail_sil_s).idxmax()]                  # clip with the most edge silence
y_raw, y_trim = A.load_wav(ex.wav_path, trim=False), A.load_wav(ex.wav_path)
t = np.arange(len(y_raw)) / A.SR

fig, ax = plt.subplots(1, 3, figsize=(17, 3.4), gridspec_kw={"width_ratios": [2, 1.4, 1.4]})
ax[0].plot(t, y_raw, lw=0.4, color="tab:gray")
ax[0].axvspan(0, ex.lead_sil_s, color="tab:red", alpha=0.25, label="removed")
ax[0].axvspan(ex.dur_s - ex.trail_sil_s, ex.dur_s, color="tab:red", alpha=0.25)
ax[0].set(title=f"{ex.id}: waveform, trimmed regions shaded", xlabel="seconds"); ax[0].legend(loc="upper right")
for a, y, name in ((ax[1], y_raw, "raw"), (ax[2], y_trim, "trimmed")):
    a.imshow(A.mel_spectrogram(y).numpy(), origin="lower", aspect="auto", cmap="magma")
    a.set(title=f"log-mel, {name} ({len(y) / A.SR:.2f} s)", xlabel="frame", ylabel="mel band")
plt.tight_layout(); savefig("1_1_E_trim_example"); plt.show()

removed = meta.dur_s - meta.trimmed_dur_s                                     # seconds removed per clip (from §1.1.7)
banner("1.1.10.2 Trimming effect over all 13,100 clips")
kv("Silence removed per clip (mean)", f"{removed.mean():.3f} s", f"max {removed.max():.2f} s")
kv("Total silence removed", f"{removed.sum() / 60:.1f} min", f"{removed.sum() / meta.dur_s.sum():.1%} of the audio")
kv("Shortest clip after trimming", f"{meta.trimmed_dur_s.min():.2f} s", "no clip trimmed to nothing")
kv("Largest fraction removed from one clip", f"{(removed / meta.dur_s).max():.1%}", "guard: speech is never cut")
check((removed / meta.dur_s).max() < 0.5, "no clip loses more than half its length to trimming")
print("\n  Listen: raw then trimmed")
display(Audio(y_raw, rate=A.SR)); display(Audio(y_trim, rate=A.SR))

### 1.1.10.3 Feature cache
**What this cell does:**
1. Computes the trimmed log-mel of every clip that is used (training subset, validation monitor and test) and saves each one as a float16 `.pt` file in the mel folder. Files that already exist are skipped, so a re-run is fast.
2. Writes one JSON record list per set: `{id, phoneme ids, number of frames}`. This is exactly what the training `Dataset` (§1.2.5.5) reads, so training never touches the wavs or the phonemiser.

**Output to expect:** how many mels were cached and how long it took, then one line per JSON file.

In [ ]:
# 1.1.10.3 Cache features: trimmed log-mel per clip (float16 on disk) + phoneme ids, for the clips we use
CACHE_SETS = {"train_subset": SUBSETS["train_subset"], "val_monitor": SUBSETS["val_monitor"], "test": SUBSETS["test_cache"]}
cache_ids = sorted(set().union(*CACHE_SETS.values()))                         # each clip computed once even if in two sets
by_id = meta.set_index("id")

def cache_mel(uid):
    """Trimmed log-mel for one clip -> MEL_DIR/<id>.pt (float16 halves the disk use; training casts back to float32)."""
    path = MEL_DIR / f"{uid}.pt"
    if not path.exists():
        torch.save(A.mel_spectrogram(A.load_wav(by_id.wav_path[uid])).half(), path)
    return uid

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(cache_mel, cache_ids))
frames = {uid: torch.load(MEL_DIR / f"{uid}.pt").shape[1] for uid in cache_ids}   # mel length per clip

# Record lists read by tts/data.py: everything training needs, nothing it does not
for name, ids in CACHE_SETS.items():
    records = [{"id": uid, "phonemes": T.phonemes_to_ids(PHONEMES[uid], VOCAB), "frames": frames[uid]} for uid in ids]
    (PROC_DIR / f"{name}.json").write_text(json.dumps(records))

banner("1.1.10.3 Feature cache")
kv("Mels cached", f"{len(cache_ids):,} clips in {time.perf_counter() - t0:.0f}s", str(MEL_DIR))
for name, ids in CACHE_SETS.items():
    kv(f"{name}.json", f"{len(ids):,} records (id, phoneme ids, frames)")

### 1.1.10.4 Mel normalisation statistics
**What this cell does:**
1. Loops over the **training-subset** mels only, keeping a running sum and a running sum of squares per mel band. This avoids holding all frames in memory at once.
2. Turns them into one global mean and std: mean = Σx / N and std = √(Σx² / N − mean²). It saves them to `mel_stats.json`. Training uses them as (mel − mean) / std, and inference reverses this before the vocoder.
3. Keeps a small seeded sample of frames (8 per clip) for the §1.1.11 plots.
4. Confirms that the normalised sample has mean ≈ 0 and std ≈ 1.

**Output to expect:** the mean (≈ −5.5), the std (≈ 2.1), the check on the normalised sample, and the number of frames used.

In [ ]:
# 1.1.10.4 Mel normalisation statistics: fitted on the TRAINING SUBSET only
# Running sums per mel band (memory-light), plus a seeded sample of frames for the plots in §1.1.11
band_sum, band_sq, n_frames, sample_frames = torch.zeros(80, dtype=torch.float64), torch.zeros(80, dtype=torch.float64), 0, []
rng = np.random.default_rng(SEED)
for uid in SUBSETS["train_subset"]:
    m = torch.load(MEL_DIR / f"{uid}.pt").double()                            # (80, frames); float64 for exact sums
    band_sum += m.sum(1); band_sq += (m ** 2).sum(1); n_frames += m.shape[1]
    sample_frames.append(m[:, rng.choice(m.shape[1], size=min(8, m.shape[1]), replace=False)])
# One global mean/std over all bands and frames: std = sqrt(E[x^2] - E[x]^2)
mean = band_sum.sum() / (n_frames * 80)
MEL_STATS = {"mel_mean": mean.item(), "mel_std": (band_sq.sum() / (n_frames * 80) - mean ** 2).sqrt().item()}
(PROC_DIR / "mel_stats.json").write_text(json.dumps(MEL_STATS, indent=1))
(RESULTS_DIR / "mel_stats.json").write_text(json.dumps(MEL_STATS, indent=1))

MEL_SAMPLE = torch.cat(sample_frames, dim=1)                                   # (80, ~8 frames per clip)
norm_sample = (MEL_SAMPLE - MEL_STATS["mel_mean"]) / MEL_STATS["mel_std"]
band_mean = band_sum / n_frames
band_std = (band_sq / n_frames - band_mean ** 2).sqrt()
banner("1.1.10.4 Mel normalisation (global mean/std from the training subset)")
kv("mel_mean", f"{MEL_STATS['mel_mean']:.4f}")
kv("mel_std", f"{MEL_STATS['mel_std']:.4f}")
kv("After normalisation: mean / std", f"{norm_sample.mean():.3f} / {norm_sample.std():.3f}", "zero-mean, unit-variance targets (on a frame sample)")
kv("Frames used", f"{n_frames:,} from {len(SUBSETS['train_subset']):,} clips")

## 1.1.11 Statistics after preprocessing
The same quantities as §1.1.7, now measured on the **processed** data, so the effect of each step is visible:
- duration after trimming;
- speaking rate in phonemes per second of speech;
- mel frames per clip;
- the distribution of mel values before and after normalisation.

**What this cell does:**
1. Builds a before/after table (duration, edge silence, speaking rate, text length and total hours) for all 13,100 clips, and saves it to `results/preprocessing_stats.csv`.
2. Draws a six-panel figure (`1_1_F_after_overview.png`):
   - duration before vs after trimming;
   - silence removed per clip;
   - mel frames per clip;
   - speaking rate in phonemes per second;
   - mel values before vs after normalisation;
   - the normalised per-band mean ± std.

**Output to expect:** the table and the figure. After normalisation the mel histogram should be centred on 0, with a spread of about 1.

In [ ]:
# 1.1.11 Before vs after preprocessing: summary table (saved) + plots
meta["mel_frames"] = meta.trimmed_dur_s * A.SR / A.HOP                        # frames = samples / hop
meta["phonemes_per_s"] = meta.n_phonemes / meta.trimmed_dur_s
rows = {                                                                       # quantity: (before, after)
    "duration (s)":            (meta.dur_s, meta.trimmed_dur_s),
    "edge silence (s)":        (meta.lead_sil_s + meta.trail_sil_s, pd.Series(0.0, index=meta.index)),
    "speaking rate":           (meta.chars_per_s, meta.phonemes_per_s),
    "text length (tokens)":    (meta.n_chars, meta.n_phonemes),
}
summary = pd.DataFrame({name: {"before: mean": b.mean(), "before: max": b.max(), "after: mean": a.mean(), "after: max": a.max()}
                        for name, (b, a) in rows.items()}).T
summary.loc["total audio (h)"] = [meta.dur_s.sum() / 3600, np.nan, meta.trimmed_dur_s.sum() / 3600, np.nan]
summary.to_csv(RESULTS_DIR / "preprocessing_stats.csv")
banner("1.1.11 Before vs after preprocessing (all 13,100 clips)")
print("  speaking rate: before = characters/s of raw audio, after = phonemes/s of trimmed speech")
print("  text length:   before = characters, after = phoneme tokens")
display(summary.round(3))
print("  saved → results/preprocessing_stats.csv")

fig, ax = plt.subplots(2, 3, figsize=(17, 7.5))
ax[0, 0].hist(meta.dur_s, bins=60, alpha=0.5, label="raw"); ax[0, 0].hist(meta.trimmed_dur_s, bins=60, alpha=0.5, label="trimmed")
ax[0, 0].set(title="Duration before vs after trimming", xlabel="seconds", ylabel="clips"); ax[0, 0].legend()
ax[0, 1].hist(meta.dur_s - meta.trimmed_dur_s, bins=60, color="tab:red"); ax[0, 1].set(title="Silence removed per clip", xlabel="seconds")
ax[0, 2].hist(meta.mel_frames, bins=60, color="tab:green"); ax[0, 2].set(title="Mel frames per clip (after)", xlabel="frames (11.6 ms each)")
ax[1, 0].hist(meta.phonemes_per_s, bins=60, color="tab:purple"); ax[1, 0].set(title="Speaking rate (after)", xlabel="phonemes / second of speech")
ax[1, 1].hist(MEL_SAMPLE.flatten().numpy(), bins=80, alpha=0.6, label="log-mel")
ax[1, 1].hist(norm_sample.flatten().numpy(), bins=80, alpha=0.6, label="normalised")
ax[1, 1].set(title="Mel values before vs after normalisation", xlabel="value"); ax[1, 1].legend()
ch_mean = ((band_mean - MEL_STATS["mel_mean"]) / MEL_STATS["mel_std"]).numpy()
ch_std = (band_std / MEL_STATS["mel_std"]).numpy()
ax[1, 2].plot(ch_mean, label="mean"); ax[1, 2].fill_between(range(80), ch_mean - ch_std, ch_mean + ch_std, alpha=0.3, label="± std")
ax[1, 2].set(title="Normalised mel per band (train subset)", xlabel="mel band", ylabel="value"); ax[1, 2].legend()
fig.suptitle("LJSpeech after preprocessing", fontsize=13)
plt.tight_layout(); savefig("1_1_F_after_overview"); plt.show()

## 1.1.12 Sanity summary
**What this cell does:** prints one line for every artefact the later sections depend on: counts, the split fingerprint, the cached sets, the vocabulary and the mel statistics. It then checks that each processed file exists. If this cell passes, §1.2 and §1.3 have all their inputs.

**Output to expect:** five summary lines and five PASS lines.

In [ ]:
# 1.1.12 Sanity summary
# (Optional) Put small sanity-check code here:
# - verify LJSpeech files exist
# - read metadata.csv
# - build split lists
# - print counts
banner("1.1.12 Sanity summary: everything later sections rely on")
kv("Metadata rows / wavs", f"{len(meta):,} / {len(wav_ids):,}")
kv("Split train / val / test", " / ".join(f"{len(SPLIT[k]):,}" for k in ("train", "val", "test")), f"md5 {SPLIT_MD5[:12]}…")
kv("Training subset / val monitor / test cached", " / ".join(f"{len(v):,}" for v in CACHE_SETS.values()))
kv("Vocabulary", f"{len(VOCAB)} symbols")
kv("Mel normalisation", f"mean {MEL_STATS['mel_mean']:.3f}, std {MEL_STATS['mel_std']:.3f}")
for f in ("train_subset.json", "val_monitor.json", "test.json", "vocab.json", "mel_stats.json"):
    check((PROC_DIR / f).exists(), f"processed/{f} written")

# 1.2 End-to-end TTS pipeline implementation (Rubric 1.2)

## 1.2.1 Pipeline overview (high level)
Describe the approach briefly:
- **TTS approach:** a custom pipeline. The acoustic model is **implemented and trained from scratch**: a simplified Matcha-TTS, with ideas from Glow-TTS. A **pretrained HiFi-GAN vocoder** converts mels to audio.
- **Key components:**
  - Text front-end: `normalize_text()` → espeak-ng IPA phonemes → integer IDs (§1.1.9)
  - Acoustic model:
    - Transformer **text encoder**: 6 layers, d=192, which predicts a mel "prior" μ for every phoneme.
    - **Monotonic Alignment Search** finds the phoneme ↔ frame alignment during training.
    - A conv **duration predictor** is trained on those durations.
    - A conv **mel decoder**, either **flow matching** (OT-CFM) or **regression**, with 8 or 4 residual blocks.
  - Vocoder / waveform generator: **HiFi-GAN V1, LJSpeech checkpoint** (Kong et al., 2020), loaded from the Matcha-TTS release. It is frozen and not trained by us.
- **Output:** playable `.wav` waveform, 22,050 Hz mono.

### Pipeline diagram
> *Diagram to be added (drawn separately and exported to `report/figures/pipeline.png`).*

```
TRAINING   text ─► normalise ─► phonemes ─► ids ─► Text Encoder ─► μ (per phoneme) ─┐
           wav ─► trim ─► log-mel ─► normalise ─► mel ───────────────────────────────┼─► MAS alignment ─► durations ─► Duration Predictor (loss)
                                                                                     └─► μ_y = μ expanded by durations ─► Decoder(mel | μ_y) (loss) ; prior loss(mel, μ_y)
INFERENCE  text ─► normalise ─► phonemes ─► ids ─► Text Encoder ─► μ ─► Duration Predictor ─► μ_y ─► Decoder ─► mel ─► de-normalise ─► HiFi-GAN ─► wav
```

### Detailed pipeline decisions
| Stage | Choice | Alternatives considered | Why this choice |
|---|---|---|---|
| Text representation | IPA phonemes (espeak-ng), stress kept | characters; ARPAbet (CMUdict) | Removes spelling ambiguity, so the model learns pronunciation directly. espeak covers any word, while a dictionary like CMUdict misses out-of-vocabulary words. |
| Text encoder | 6-layer Transformer, positional embeddings | CNN/LSTM encoder (Tacotron 2) | Self-attention sees the whole sentence, which gives context-dependent prosody. It is parallel, so training is fast. |
| Alignment | **Monotonic Alignment Search** (Glow-TTS), with no external aligner | attention (Tacotron 2); external forced aligner (MFA, FastSpeech 2) | Speech is monotonic: text is read left to right. MAS learns the alignment jointly, needs no extra tool, and cannot skip or repeat words, which are the classic attention failures. |
| Durations at inference | conv duration predictor on log-durations | autoregressive attention | Fully parallel, so it is fast. Speaking rate can be controlled with `length_scale`. |
| Mel decoder | **flow matching** (OT-CFM, Matcha-TTS) **vs regression** (FastSpeech-style), conv residual blocks | diffusion (Grad-TTS); U-Net + Transformer decoder (full Matcha) | Flow matching is a generative model that needs only a few ODE steps, unlike diffusion's hundreds. Regression is the simplest baseline. Comparing the two is our controlled experiment. The 1-D conv backbone is small enough to train on the full training split in about 2 hours on a T4. |
| Vocoder | **pretrained HiFi-GAN V1 (LJSpeech)** | Griffin-Lim; training our own vocoder; WaveGlow | Griffin-Lim sounds robotic, and training a GAN vocoder takes days. HiFi-GAN is fast (RTF ≪ 1) and high quality. Its mel settings define ours (§1.1.10). |
| Training data | **full training split (10,480 utterances)**, 300 epochs | 3,000-utterance subset (task sheet: 1–3k) | Our 3,000-utterance runs over-fitted the duration predictor (train 0.41 vs val 0.55), giving flat, robotic rhythm. The full split fixes this and still fits four runs on free Colab GPUs. |

*The architecture follows Matcha-TTS (Mehta et al., 2024) and Glow-TTS (Kim et al., 2020), simplified. Module code was written by our team; adapted parts are credited in each file header.*

## 1.2.2 Environment & dependencies (reproducible)
Provide at least one dependency definition file and document:
- **OS:** Ubuntu (Google Colab) for training; macOS 26 (Apple M4) for development. The exact values are printed in §0.2.
- **Python:** 3.13 (§0.2)
- **CUDA (if used):** printed in §0.2 (`results/environment.json`)
- **GPU (if used):** Colab GPU, printed in §0.2

✅ Include: `requirements.txt` or `environment.yml` (or equivalent)
→ `requirements.txt` at the repository root. espeak-ng is a system package: `apt install espeak-ng` on Linux, `brew install espeak-ng` on macOS.

## 1.2.3 Training command(s)
Provide the exact command(s) you used, for example:
- `python scripts/train.py --config configs/expA.yaml`
- `python scripts/train.py --config configs/expB.yaml`

If you use a framework launcher, include that too (e.g., `accelerate launch ...`).

**Ours:**
- In this notebook: `train(load_config("expA"))` in §1.3. It is the same function for all four experiments.
- From a terminal: `python -m tts.train --config configs/expA.json --proc_dir <processed dir> --mel_dir <mel dir> --ckpt_root checkpoints/runs`

## 1.2.4 Inference: a single “text → wav” entrypoint (must be easy to run)
Provide a minimal command that generates a waveform file from text, for example:
- `python scripts/synth.py --text "Hello world." --checkpoint checkpoints/expA.ckpt --out wav_examples/hello.wav`

Also state:
- Where checkpoints are stored / how to download them (if applicable)
- Where generated wavs are saved by default

**Ours:**
- **Python:** `synthesize_to_wav("Hello world.", exp="expA", out_path="wav_examples/hello.wav")`
- **Terminal:** `python scripts/synth.py --text "Hello world." --exp expA --out wav_examples/hello.wav`
- **Checkpoints:** `checkpoints/runs/<exp>/best.ckpt`, plus `last.ckpt` and milestones `epoch_XXX.ckpt`. Each checkpoint embeds its config, vocabulary and mel statistics, so it is self-contained.
- **Vocoder:** `checkpoints/hifigan_lj_v1`, downloaded automatically.
- **Output:** wavs go to `wav_examples/` by default.

---
### Code in §1.2, in run order
| # | Cell | Writes / does |
|---|---|---|
| 1.2.5.1 | Monotonic Alignment Search | `tts/align.py` |
| 1.2.5.2 | Text encoder & duration predictor | `tts/model.py` |
| 1.2.5.3 | Flow-matching and regression decoders | `tts/decoders.py` |
| 1.2.5.4 | Full acoustic model & losses | `tts/tts_model.py` |
| 1.2.5.5 | Dataset & batching | `tts/data.py` |
| 1.2.5.6 | Training loop & checkpointing | `tts/train.py` |
| 1.2.6.1 | Pretrained vocoder | `tts/hifigan/`, `checkpoints/hifigan_lj_v1` |
| 1.2.6.2 | Inference module | `tts/synth.py` |
| 1.2.6.3 | Command-line script | `scripts/synth.py` |
| 1.2.7 | Module walk-through | tensor shapes and parameter counts |
| 1.2.8 | Pipeline wiring | the text → wav entry point |

## 1.2.5 Model code (the `tts/` package)
The six cells below each write one module with `%%writefile`. Because the code is written to files, the notebook, the training CLI and `scripts/synth.py` all run **the same code**, and the marker can still read all of it here. Each cell prints `Writing tts/<file>.py` (or `Overwriting` on a re-run). Nothing is executed until §1.2.7.

### 1.2.5.1 Monotonic Alignment Search (`tts/align.py`)
**The problem it solves:** LJSpeech says *what* was said but not *when*. Training needs to know which mel frames belong to which phoneme.

**What the module does:**
- `log_likelihood()` scores every (frame, phoneme) pair: how close is this mel frame to that phoneme's prior μ?
- `maximum_path()` finds the best **monotonic** path through that score matrix by dynamic programming (`_mas_single`, compiled with Numba for speed). On that path every frame belongs to exactly one phoneme, phonemes are used left to right, and none is skipped.
- Summing the path over frames gives each phoneme's **duration**, which is the training target for the duration predictor.

MAS runs without gradients. As μ improves during training, the alignment improves with it.

In [ ]:
%%writefile tts/align.py
"""Monotonic Alignment Search (MAS), Glow-TTS (Kim et al., 2020):
https://github.com/jaywalnut310/glow-tts/blob/master/monotonic_align/core.pyx  (ported from Cython to Numba)."""
import numpy as np, torch
from numba import njit


@njit(cache=True)
def _mas_single(value, t_y, t_x):
    """value: (t_y frames, t_x phonemes) log-likelihoods, overwritten in place with cumulative best scores.
    Returns the 0/1 alignment path of the same shape."""
    path = np.zeros((t_y, t_x), dtype=np.float32)
    neg = -1e9
    for y in range(t_y):                                     # forward: best cumulative score per (frame, phoneme)
        # x is limited so that every phoneme still gets at least one of the remaining frames
        for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
            v_cur = neg if x == y else value[y - 1, x]       # stay on the same phoneme
            v_prev = (0.0 if y == 0 else neg) if x == 0 else value[y - 1, x - 1]   # move to the next phoneme
            value[y, x] += max(v_prev, v_cur)
    index = t_x - 1                                          # backward: trace the best path
    for y in range(t_y - 1, -1, -1):
        path[y, index] = 1.0
        if index != 0 and (index == y or value[y - 1, index] < value[y - 1, index - 1]):
            index -= 1
    return path


@torch.no_grad()
def log_likelihood(mu, mel):
    """mu: (B, T_text, 80), mel: (B, 80, T_mel) -> (B, T_mel, T_text) Gaussian log-likelihood (up to a constant)."""
    # log N(frame | mu, I) = -0.5 * ||frame - mu||^2 + const, for every frame/phoneme pair at once
    return -0.5 * torch.cdist(mel.transpose(1, 2), mu) ** 2


@torch.no_grad()
def maximum_path(score, text_lens, mel_lens):
    """0/1 path (B, T_mel, T_text): path[b, j, i] = 1 if mel frame j belongs to phoneme i."""
    score_np = score.float().cpu().numpy()                   # Numba runs on CPU numpy arrays
    path = np.zeros_like(score_np)
    for b in range(score_np.shape[0]):                       # one clip at a time, ignoring its padding
        ty, tx = int(mel_lens[b]), int(text_lens[b])
        path[b, :ty, :tx] = _mas_single(score_np[b, :ty, :tx].copy(), ty, tx)
    return torch.from_numpy(path).to(score.device)

### 1.2.5.2 Text encoder and duration predictor (`tts/model.py`)
**What the module does:**
- `TextEncoder`:
  - embeds each phoneme ID and adds a learned position embedding;
  - runs 6 Transformer layers, with padding masked out, so every phoneme sees the whole sentence;
  - projects each phoneme to an 80-dimensional **mel prior μ**, the model's first guess of what that phoneme's frames look like.
- `DurationPredictor`:
  - three 1-D convolutions that predict **log(number of frames)** per phoneme from the encoder states;
  - its input is *detached*, so the duration loss trains only the predictor and does not pull on the encoder.

In [ ]:
%%writefile tts/model.py
"""Text encoder + duration predictor, after Glow-TTS (Kim et al., 2020) and Matcha-TTS (Mehta et al., 2024)."""
import torch, torch.nn as nn
from tts.text import PAD_ID


class TextEncoder(nn.Module):
    """Phoneme ids -> hidden states h and a mel prior mu for every phoneme."""
    def __init__(self, n_vocab, d_model=192, n_layers=6, n_heads=2, n_mels=80, max_len=1000):
        super().__init__()
        self.embed = nn.Embedding(n_vocab, d_model, padding_idx=PAD_ID)
        self.pos = nn.Embedding(max_len, d_model)                            # learned position embedding
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=4 * d_model, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.proj_mu = nn.Linear(d_model, n_mels)                            # hidden state -> mel prior

    def forward(self, ids):
        pad_mask = ids == PAD_ID                                             # True on padding
        x = self.embed(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))
        h = self.transformer(x, src_key_padding_mask=pad_mask)
        return h, self.proj_mu(h), pad_mask                                  # mu: (B, T_text, 80)


class DurationPredictor(nn.Module):
    """Predicts log(frames) per phoneme from the (detached) encoder states."""
    def __init__(self, d_model=192, hidden=256, kernel=3, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(d_model, hidden, kernel, padding=kernel // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, kernel, padding=kernel // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, 1, 1))

    def forward(self, h, pad_mask):
        logw = self.net(h.detach().transpose(1, 2)).squeeze(1)              # (B, T_text)
        return logw.masked_fill(pad_mask, 0.0)

### 1.2.5.3 The two decoders (`tts/decoders.py`): the experimental variable
**What the module does:** both decoders turn **μ_y** (the phoneme priors stretched to frame level by the durations) into the final mel. Both use the same `_Backbone`: a 1×1 input conv, then `n_blocks` residual conv blocks with dilations 1, 2, 4, 8, 1, 2, … and 256 channels, then a 1×1 output conv. So **only the training objective differs** between them.
- **`FlowMatchingDecoder`** (expA, expB), a generative model:
  - *Training:* pick Gaussian noise x₀, a random time t ∈ [0, 1] and the point x_t on the straight line from x₀ to the real mel x₁. The network, given x_t, μ_y and t, learns the **velocity** that points from x₀ to x₁.
  - *Sampling:* start from noise scaled by `temperature` and take `n_steps` Euler steps along the learned velocity. That is `n_steps` network passes.
- **`RegressionDecoder`** (expC, expD), the baseline:
  - predicts the mel directly from μ_y in **one pass**, trained with masked L1 loss;
  - no noise and no time input, so the output is deterministic.

`n_blocks` (8 or 4) is the **depth** variable of the experiments. `build_decoder("flow" | "regression", ...)` picks the class from the config.

In [ ]:
%%writefile tts/decoders.py
"""Two mel decoders sharing one backbone, so the ONLY difference between them is the training objective.

FlowMatchingDecoder: OT-CFM (Lipman et al., 2023) as in Matcha-TTS (Mehta et al., 2024): learns a velocity field that
    moves Gaussian noise to the mel, conditioned on mu_y; sampled with n Euler ODE steps.
RegressionDecoder: FastSpeech-style (Ren et al., 2021): predicts the mel directly from mu_y with an L1 loss, one pass.
Backbone: 1-D residual conv blocks with dilations 1,2,4,8 (a simplification of Matcha's U-Net + Transformer).
"""
import math, torch, torch.nn as nn, torch.nn.functional as F


class TimeEmbedding(nn.Module):
    """Sinusoidal embedding of the flow time t in [0, 1] -> vector."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = 1000 * t[:, None] * freqs[None]
        return self.mlp(torch.cat([args.sin(), args.cos()], dim=1))


class ResBlock(nn.Module):
    """Pre-norm residual block: norm -> SiLU -> dilated conv (+ time embedding) -> norm -> SiLU -> conv, plus skip.
    Multiplying by the mask after every conv keeps padded frames at exactly zero."""
    def __init__(self, ch, dilation, time_cond):
        super().__init__()
        self.norm1, self.norm2 = nn.GroupNorm(8, ch), nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv1d(ch, ch, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=1)
        self.time = nn.Linear(ch, ch) if time_cond else None

    def forward(self, x, mask, temb=None):
        y = self.conv1(F.silu(self.norm1(x))) * mask
        if self.time is not None:                                                # flow decoder only: tell the block "where" on the path it is
            y = y + self.time(temb)[:, :, None]
        y = self.conv2(F.silu(self.norm2(y))) * mask
        return (x + y) * mask


class _Backbone(nn.Module):
    """Shared by both decoders: 1x1 conv in -> n_blocks ResBlocks (dilation 1,2,4,8 repeating) -> 1x1 conv out."""
    def __init__(self, in_ch, n_mels, ch, n_blocks, time_cond):
        super().__init__()
        self.inp = nn.Conv1d(in_ch, ch, 1)
        self.blocks = nn.ModuleList([ResBlock(ch, 2 ** (i % 4), time_cond) for i in range(n_blocks)])
        self.out = nn.Conv1d(ch, n_mels, 1)

    def forward(self, x, mask, temb=None):
        h = self.inp(x) * mask
        for blk in self.blocks:
            h = blk(h, mask, temb)
        return self.out(h) * mask


class FlowMatchingDecoder(nn.Module):
    kind = "flow"

    def __init__(self, n_mels=80, ch=256, n_blocks=8, sigma_min=1e-4):
        super().__init__()
        self.sigma_min = sigma_min
        self.time = TimeEmbedding(ch)
        self.net = _Backbone(2 * n_mels, n_mels, ch, n_blocks, time_cond=True)   # input: [x_t ; mu_y]

    def forward(self, x, mask, mu_y, t):
        return self.net(torch.cat([x, mu_y], dim=1), mask, self.time(t))

    def loss(self, x1, mask, mu_y):
        """OT-CFM loss: MSE between the predicted and the true velocity at a random point of the noise->mel path."""
        x0 = torch.randn_like(x1)                                                # noise
        t = torch.rand(x1.shape[0], device=x1.device)[:, None, None]             # one time per clip
        x_t = (1 - (1 - self.sigma_min) * t) * x0 + t * x1                       # point on the straight path
        u = x1 - (1 - self.sigma_min) * x0                                       # target velocity
        v = self(x_t, mask, mu_y, t[:, 0, 0])
        return ((v - u) ** 2 * mask).sum() / (mask.sum() * x1.shape[1])

    @torch.no_grad()
    def sample(self, mu_y, mask, n_steps=10, temperature=0.667):
        """Integrate the learned velocity from t=0 (noise) to t=1 (mel). temperature < 1 reduces sampling noise."""
        x = torch.randn_like(mu_y) * temperature
        dt = 1.0 / n_steps
        for i in range(n_steps):                                                 # Euler ODE solver
            t = torch.full((mu_y.shape[0],), i * dt, device=mu_y.device)
            x = x + dt * self(x, mask, mu_y, t)
        return x * mask


class RegressionDecoder(nn.Module):
    kind = "regression"

    def __init__(self, n_mels=80, ch=256, n_blocks=8):
        super().__init__()
        self.net = _Backbone(n_mels, n_mels, ch, n_blocks, time_cond=False)      # input: mu_y only

    def forward(self, mask, mu_y):
        return self.net(mu_y, mask)

    def loss(self, x1, mask, mu_y):
        return ((self(mask, mu_y) - x1).abs() * mask).sum() / (mask.sum() * x1.shape[1])   # masked L1

    @torch.no_grad()
    def sample(self, mu_y, mask, **_):
        return self(mask, mu_y)                                                  # deterministic, one pass


DECODERS = {"flow": FlowMatchingDecoder, "regression": RegressionDecoder}   # config value "decoder" -> class


def build_decoder(kind, **kwargs):
    return DECODERS[kind](**kwargs)

### 1.2.5.4 Full acoustic model and losses (`tts/tts_model.py`)
**What the module does:** `TTSModel` joins the encoder, the duration predictor and one decoder.
- **Training (`forward`)**, in order:
  1. encode the phonemes → μ;
  2. MAS alignment → durations;
  3. **duration loss:** MSE between the predicted and the MAS log-durations;
  4. expand μ to frame level → μ_y;
  5. **prior loss:** Gaussian negative log-likelihood of the real mel under μ_y. It pulls μ towards the real frames, which also sharpens the next alignment;
  6. **decoder loss** from §1.2.5.3.
  - Total loss = the sum of the three. The three parts are also returned for logging.
- **Inference (`synthesise_mel`):**
  1. encode;
  2. predict durations (exp, scaled by `length_scale`, rounded up);
  3. expand to μ_y;
  4. sample the decoder.

In [ ]:
%%writefile tts/tts_model.py
"""Full acoustic model and losses, following Matcha-TTS (Mehta et al., 2024) and Glow-TTS (Kim et al., 2020).
loss = duration loss (MSE on log-durations) + prior loss (Gaussian NLL of mel under mu_y) + decoder loss."""
import math, torch, torch.nn as nn
from tts.model import TextEncoder, DurationPredictor
from tts.decoders import build_decoder
from tts.align import log_likelihood, maximum_path


def length_mask(lens, max_len):
    """(B,) lengths -> (B, max_len) mask: 1 on real frames, 0 on padding."""
    return (torch.arange(max_len, device=lens.device)[None] < lens[:, None]).float()


def durations_to_path(d, t_mel):
    """d: (T_text,) integer durations -> 0/1 path (T_mel, T_text), the same format MAS produces."""
    end = torch.cumsum(d, 0)
    frames = torch.arange(t_mel, device=d.device)[:, None]
    return ((frames >= (end - d)[None]) & (frames < end[None])).float()


class TTSModel(nn.Module):
    def __init__(self, n_vocab, decoder="flow", dec_blocks=8, n_mels=80, dec_channels=256):
        super().__init__()
        self.encoder = TextEncoder(n_vocab, n_mels=n_mels)
        self.dur = DurationPredictor()
        self.decoder = build_decoder(decoder, n_mels=n_mels, ch=dec_channels, n_blocks=dec_blocks)

    def forward(self, ids, ids_lens, mels, mel_lens):
        """Training step: returns the total loss and its three parts."""
        # 1. Encode text; masks mark real phonemes (x_mask) and real mel frames (y_mask)
        h, mu, pad_mask = self.encoder(ids)
        x_mask = (~pad_mask).float()
        y_mask = length_mask(mel_lens, mels.shape[2])

        # 2. Alignment: which frames belong to which phoneme
        path = maximum_path(log_likelihood(mu, mels), ids_lens, mel_lens)        # alignment, no gradients
        durations = path.sum(dim=1)                                              # frames per phoneme

        # 3. Duration loss: predicted vs MAS log-durations, averaged over real phonemes
        logw = self.dur(h, pad_mask)
        dur_loss = ((logw - torch.log(durations + 1e-8) * x_mask) ** 2).sum() / ids_lens.sum()

        # 4. Prior loss: copy each phoneme's mu to its frames, then Gaussian NLL of the real mel under it
        mu_y = (path @ mu).transpose(1, 2)                                       # (B, 80, T_mel)
        prior_loss = (0.5 * ((mels - mu_y) ** 2 + math.log(2 * math.pi)) * y_mask[:, None]).sum() \
                     / (y_mask.sum() * mels.shape[1])
        # 5. Decoder loss: flow matching or regression, conditioned on mu_y
        dec_loss = self.decoder.loss(mels, y_mask[:, None], mu_y)
        total = dur_loss + prior_loss + dec_loss
        return total, {"dur": dur_loss.item(), "prior": prior_loss.item(), "dec": dec_loss.item()}

    @torch.no_grad()
    def synthesise_mel(self, ids, n_steps=10, temperature=0.667, length_scale=1.0):
        """ids: (1, T_text) -> normalised mel (1, 80, T_mel) and integer durations (T_text,)."""
        h, mu, pad_mask = self.encoder(ids)
        logw = self.dur(h, pad_mask)[0]
        d = torch.ceil(torch.exp(logw) * length_scale).long().clamp(min=1)      # frames per phoneme, at least 1
        t_mel = int(d.sum())
        mu_y = (durations_to_path(d, t_mel) @ mu[0]).T[None]
        mask = torch.ones(1, 1, t_mel, device=ids.device)
        return self.decoder.sample(mu_y, mask, n_steps=n_steps, temperature=temperature), d

### 1.2.5.5 Dataset and batching (`tts/data.py`)
**What the module does:**
- `TTSDataset` reads a record list from §1.1.10.3. For each item it loads the cached mel and normalises it with the **training** mean/std, then returns (phoneme IDs, mel).
- `collate()` pads a batch to its longest item: phoneme IDs with 0 (`<pad>`) and mels with zeros. It returns the true lengths, so the masks in the model ignore the padding.

In [ ]:
%%writefile tts/data.py
"""Dataset over the cached features of §1.1.10: (phoneme ids, normalised log-mel)."""
import json, torch
from torch.utils.data import Dataset
from tts.text import PAD_ID


class TTSDataset(Dataset):
    def __init__(self, records_json, mel_dir, mel_stats):
        with open(records_json) as f:
            self.records = json.load(f)
        self.mel_dir, self.mean, self.std = mel_dir, mel_stats["mel_mean"], mel_stats["mel_std"]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        mel = torch.load(f"{self.mel_dir}/{r['id']}.pt").float()
        return torch.tensor(r["phonemes"], dtype=torch.long), (mel - self.mean) / self.std


def collate(batch):
    """List of (ids, mel) -> padded ids (B, T_text), true text lengths, padded mels (B, 80, T_mel), true mel lengths."""
    ids_list, mel_list = zip(*batch)
    ids_lens = torch.tensor([len(x) for x in ids_list])
    mel_lens = torch.tensor([m.shape[1] for m in mel_list])
    ids = torch.full((len(batch), int(ids_lens.max())), PAD_ID, dtype=torch.long)
    mels = torch.zeros(len(batch), mel_list[0].shape[0], int(mel_lens.max()))
    for i, (x, m) in enumerate(zip(ids_list, mel_list)):
        ids[i, :len(x)] = x
        mels[i, :, :m.shape[1]] = m
    return ids, ids_lens, mels, mel_lens

### 1.2.5.6 Training loop and checkpointing (`tts/train.py`)
**What the module does:** `train(cfg)` is one function used by **all four** experiments; only the config changes.
1. **Setup:** seeds everything, picks the device, and builds a seeded shuffling DataLoader for the training subset and a fixed one for the validation monitor. Then builds the model from `cfg["decoder"]` and `cfg["dec_blocks"]`, and the Adam optimiser.
2. **Resume:** if `last.ckpt` exists, it restores the model, the optimiser, the log and the RNG state, and continues from the next epoch.
3. **Each epoch:**
   - for every batch: forward → loss → backward → gradient clipping (1.0) → Adam step;
   - then `evaluate()` scores the validation-monitor clips. The flow-matching loss draws random noise and t, so the RNG is fixed during evaluation: every epoch is scored on the *same* noise, which keeps validation losses comparable between epochs.
4. **Checkpoints:**
   - `last.ckpt` every epoch, to resume;
   - `best.ckpt` when the validation loss improves;
   - `epoch_XXX.ckpt` at the milestones.
   - Each checkpoint also stores the config, the vocabulary and the mel statistics, so it can be used alone at inference.
5. **Log:** `train_log.csv` is rewritten every epoch, and a progress line is printed every `verbose_every` epochs.

It can also be run from a terminal: `python -m tts.train --config ...`.

In [ ]:
%%writefile tts/train.py
"""Training loop shared by every experiment: fixed seed, resume from last.ckpt, best.ckpt by validation loss,
milestone checkpoints, per-epoch CSV log.   CLI: python -m tts.train --config configs/expA.json ..."""
import argparse, json, os, random, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from tts.data import TTSDataset, collate
from tts.tts_model import TTSModel


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def pick_device():
    return torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


@torch.no_grad()
def evaluate(model, loader, device, seed):
    """Mean validation losses. The flow loss samples noise and t, so the RNG is fixed: every epoch is scored on the
    same noise and the scores are comparable across epochs."""
    model.eval()                                                             # disables dropout
    # fork_rng: the fixed seed below does not disturb the training RNG stream
    kw = {"device_type": device.type} if device.type in ("cuda", "mps") else {"devices": []}
    totals, n = {"dur": 0.0, "prior": 0.0, "dec": 0.0}, 0
    with torch.random.fork_rng(**kw):
        torch.manual_seed(seed)
        for batch in loader:
            _, parts = model(*[x.to(device) for x in batch])
            for k in totals:
                totals[k] += parts[k]
            n += 1
    model.train()
    return {k: v / n for k, v in totals.items()}


def train(cfg, verbose_every=10):
    """cfg: experiment config + paths (proc_dir, mel_dir, ckpt_dir). Returns the per-epoch log as a DataFrame."""
    # ---- 1. Setup: seed, device, data, model, optimiser ----
    set_seed(cfg["seed"])
    device = pick_device()
    ckpt_dir = Path(cfg["ckpt_dir"]); ckpt_dir.mkdir(parents=True, exist_ok=True)
    proc = Path(cfg["proc_dir"])
    vocab = json.loads((proc / "vocab.json").read_text())
    mel_stats = json.loads((proc / "mel_stats.json").read_text())

    workers = 2 if device.type == "cuda" else 0                              # background loading only helps on CUDA
    g = torch.Generator().manual_seed(cfg["seed"])                           # same shuffle order in every experiment
    train_dl = DataLoader(TTSDataset(proc / "train_subset.json", cfg["mel_dir"], mel_stats), batch_size=cfg["batch_size"],
                          shuffle=True, collate_fn=collate, num_workers=workers, drop_last=True, generator=g)
    val_dl = DataLoader(TTSDataset(proc / "val_monitor.json", cfg["mel_dir"], mel_stats), batch_size=cfg["batch_size"],
                        shuffle=False, collate_fn=collate, num_workers=workers)

    model = TTSModel(len(vocab), decoder=cfg["decoder"], dec_blocks=cfg["dec_blocks"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    milestones = set(cfg["milestones"])

    # ---- 2. Resume from last.ckpt if this experiment was interrupted ----
    start, best_val, best_epoch, log, elapsed = 1, float("inf"), 0, [], 0.0
    last_path = ckpt_dir / "last.ckpt"
    if last_path.exists():
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best_val, best_epoch, log, elapsed = ck["epoch"] + 1, ck["best_val"], ck["best_epoch"], ck["log"], ck["elapsed_s"]
        torch.set_rng_state(ck["rng_cpu"].cpu())                                     # map_location moved it off the CPU
        print(f"  resumed {cfg['name']} from epoch {ck['epoch']} (best so far {best_val:.4f} @ epoch {best_epoch})")
    if start > cfg["epochs"]:
        print(f"  {cfg['name']}: already trained for {cfg['epochs']} epochs, nothing to do")
        return pd.DataFrame(log)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {cfg['name']}: decoder={cfg['decoder']}, blocks={cfg['dec_blocks']}, params={n_params / 1e6:.2f}M, "
          f"{len(train_dl)} steps/epoch, device={device}")
    # ---- 3. Epoch loop: train on every batch, then score the validation monitor ----
    for epoch in range(start, cfg["epochs"] + 1):
        t0, running = time.time(), []
        for batch in train_dl:
            total, parts = model(*[x.to(device) for x in batch])
            opt.zero_grad()
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])   # guards against rare large gradients
            opt.step()
            running.append([total.item(), parts["dur"], parts["prior"], parts["dec"]])

        val = evaluate(model, val_dl, device, cfg["seed"])
        val_total = sum(val.values())                                        # same sum of three parts as the training loss
        is_best = val_total < best_val
        if is_best:
            best_val, best_epoch = val_total, epoch
        sec = time.time() - t0
        elapsed += sec
        tr = np.mean(running, axis=0)
        log.append({"epoch": epoch, "train": tr[0], "train_dur": tr[1], "train_prior": tr[2], "train_dec": tr[3],
                    "val": val_total, **{f"val_{k}": v for k, v in val.items()}, "best": is_best, "sec": round(sec, 1)})

        # ---- 4. Checkpoints (self-contained: config, vocab and mel stats travel with the weights) + CSV log ----
        ck = {"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best_val": best_val,
              "best_epoch": best_epoch, "log": log, "elapsed_s": elapsed, "cfg": cfg, "vocab": vocab,
              "mel_stats": mel_stats, "n_vocab": len(vocab), "rng_cpu": torch.get_rng_state()}
        torch.save(ck, last_path)
        if is_best:
            torch.save(ck, ckpt_dir / "best.ckpt")
        if epoch in milestones:
            torch.save(ck, ckpt_dir / f"epoch_{epoch:03d}.ckpt")
        pd.DataFrame(log).to_csv(ckpt_dir / "train_log.csv", index=False)

        # ---- 5. Progress line: ★ = new best.ckpt, ◆ = milestone checkpoint written ----
        if epoch == start or epoch % verbose_every == 0 or epoch == cfg["epochs"]:
            print(f"  [{cfg['name']}] epoch {epoch:>3}/{cfg['epochs']}  train {tr[0]:.4f}  val {val_total:.4f} "
                  f"(dur {val['dur']:.3f} prior {val['prior']:.3f} dec {val['dec']:.3f})  {sec:.1f}s/epoch"
                  f"{'  ★ best' if is_best else ''}{'  ◆ milestone saved' if epoch in milestones else ''}")
    print(f"  {cfg['name']} done: best val {best_val:.4f} at epoch {best_epoch}, total {elapsed / 60:.1f} min")
    return pd.DataFrame(log)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Train one experiment from a config file.")
    ap.add_argument("--config", required=True)
    ap.add_argument("--proc_dir", required=True)
    ap.add_argument("--mel_dir", required=True)
    ap.add_argument("--ckpt_root", default="checkpoints/runs")
    a = ap.parse_args()
    c = json.loads(Path(a.config).read_text())
    train({**c, "proc_dir": a.proc_dir, "mel_dir": a.mel_dir, "ckpt_dir": str(Path(a.ckpt_root) / c["name"])})

## 1.2.6 Vocoder and inference code
### 1.2.6.1 Pretrained HiFi-GAN V1 (LJSpeech)
We use the official HiFi-GAN V1 generator trained on LJSpeech (Kong et al., 2020) as distributed with Matcha-TTS. The model code comes from `matcha/hifigan` (MIT licence) and is copied to `tts/hifigan/`. The weights are downloaded once to `checkpoints/hifigan_lj_v1`.

**Caveat for the report:** this vocoder was trained on all of LJSpeech, including our test clips. It only converts mel → waveform and is identical for all four experiments, so the comparison is still fair.

**What the cell below does:**
1. If `tts/hifigan/` is missing, shallow-clones Matcha-TTS into the work folder and copies its `hifigan/` code.
2. If the weights are missing, downloads the generator checkpoint.

Both steps are skipped when the files already exist.

**Output to expect:** the code location and the size of the weights file.

In [ ]:
# 1.2.6.1 Vocoder setup: copy HiFi-GAN code from Matcha-TTS (MIT) and download the LJ V1 generator weights
import urllib.request
MATCHA_DIR = WORK_DIR / "Matcha-TTS"
if not (TTS_DIR / "hifigan" / "models.py").exists():
    if not MATCHA_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/shivammehta25/Matcha-TTS.git", str(MATCHA_DIR)], check=True)
    shutil.copytree(MATCHA_DIR / "matcha" / "hifigan", TTS_DIR / "hifigan", dirs_exist_ok=True)
if not VOCODER_PATH.exists():
    urllib.request.urlretrieve("https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/generator_v1", VOCODER_PATH)

banner("1.2.6.1 Vocoder")
kv("HiFi-GAN code", TTS_DIR.relative_to(PROJECT_ROOT) / "hifigan", "from shivammehta25/Matcha-TTS (MIT)")
kv("HiFi-GAN LJ V1 weights", VOCODER_PATH.relative_to(PROJECT_ROOT), f"{VOCODER_PATH.stat().st_size / 1e6:.1f} MB")

### 1.2.6.2 Inference module (`tts/synth.py`)
**What the module does:**
- `load_vocoder()`: builds the HiFi-GAN V1 generator, loads the weights, and removes weight-norm, which makes inference faster.
- `load_tts()`: rebuilds an acoustic model from a checkpoint. The checkpoint stores its own config, so the right decoder type and depth are built automatically.
- `synthesize()`: text → phoneme IDs → normalised mel → **de-normalise** (× std + mean) → HiFi-GAN → waveform clipped to [−1, 1].
- `synthesize_to_wav()`: the one-call entry point. It caches loaded models, so repeated calls are fast, and writes the `.wav`.
- `main()`: the command-line interface used by `scripts/synth.py`.

In [ ]:
%%writefile tts/synth.py
"""Inference: text -> phonemes -> acoustic model -> mel -> HiFi-GAN -> wav.
HiFi-GAN (Kong et al., 2020), code + LJ V1 weights via Matcha-TTS: https://github.com/shivammehta25/Matcha-TTS
CLI: python -m tts.synth --text "Hello world." --exp expA --out wav_examples/hello.wav"""
import argparse, time
from pathlib import Path
import numpy as np, soundfile as sf, torch
from tts.hifigan.config import v1
from tts.hifigan.env import AttrDict
from tts.hifigan.models import Generator
from tts.text import text_to_ids, normalize_text
from tts.tts_model import TTSModel
from tts.train import pick_device

SR = 22050
_CACHE = {}                                                          # (checkpoint, device) -> loaded model; ("voc", device) -> vocoder


def load_vocoder(path, device):
    voc = Generator(AttrDict(v1)).to(device)
    voc.load_state_dict(torch.load(path, map_location=device, weights_only=False)["generator"])
    voc.eval(); voc.remove_weight_norm()
    return voc


def load_tts(ckpt_path, device):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = TTSModel(ck["n_vocab"], decoder=ck["cfg"]["decoder"], dec_blocks=ck["cfg"]["dec_blocks"]).to(device)
    model.load_state_dict(ck["model"]); model.eval()
    return model, ck


@torch.no_grad()
def synthesize(text, model, vocoder, vocab, mel_stats, n_steps=10, temperature=0.667, length_scale=1.0):
    """Returns (wav float32 numpy, sample rate, normalised mel tensor)."""
    device = next(model.parameters()).device
    ids = torch.tensor([text_to_ids(text, vocab)], device=device)
    mel, _ = model.synthesise_mel(ids, n_steps=n_steps, temperature=temperature, length_scale=length_scale)
    # undo the §1.1.10.4 normalisation: HiFi-GAN expects raw log-mels
    wav = vocoder(mel * mel_stats["mel_std"] + mel_stats["mel_mean"]).squeeze().clamp(-1, 1).cpu().numpy()
    return wav.astype(np.float32), SR, mel


def synthesize_to_wav(text, exp="expA", out_path=None, ckpt="best", ckpt_root="checkpoints/runs",
                      vocoder_path="checkpoints/hifigan_lj_v1", **kw):
    """One call from text to a playable .wav: the notebook / CLI entry point."""
    device = pick_device()
    key = (str(Path(ckpt_root) / exp / f"{ckpt}.ckpt"), str(device))
    if key not in _CACHE:
        model, ck = load_tts(key[0], device)
        if ("voc", str(device)) not in _CACHE:
            _CACHE[("voc", str(device))] = load_vocoder(vocoder_path, device)
        _CACHE[key] = (model, ck)
    model, ck = _CACHE[key]
    wav, sr, mel = synthesize(text, model, _CACHE[("voc", str(device))], ck["vocab"], ck["mel_stats"], **kw)
    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_path, wav, sr)
    return wav, sr


def main():
    ap = argparse.ArgumentParser(description="Synthesise a .wav from text with a trained experiment.")
    ap.add_argument("--text", required=True)
    ap.add_argument("--exp", default="expA", help="expA | expB | expC | expD")
    ap.add_argument("--ckpt", default="best", help="best | last | epoch_XXX")
    ap.add_argument("--out", default="wav_examples/output.wav")
    ap.add_argument("--ckpt_root", default="checkpoints/runs")
    ap.add_argument("--vocoder", default="checkpoints/hifigan_lj_v1")
    ap.add_argument("--steps", type=int, default=10, help="ODE steps (flow-matching decoders only)")
    a = ap.parse_args()
    t0 = time.perf_counter()
    wav, sr = synthesize_to_wav(a.text, a.exp, a.out, a.ckpt, a.ckpt_root, a.vocoder, n_steps=a.steps)
    print(f"normalised text : {normalize_text(a.text)}")
    print(f"wrote {a.out}: {len(wav) / sr:.2f} s of audio in {time.perf_counter() - t0:.2f} s")


if __name__ == "__main__":
    main()

### 1.2.6.3 Command-line script (`scripts/synth.py`)
**What the script does:** it is a thin wrapper, so the marker can run `python scripts/synth.py --text "..." --exp expA --out wav_examples/x.wav` from the repository root. It puts the repository on the import path, moves to the repository root, points phonemizer at espeak-ng on macOS, and then calls `tts.synth.main()`. All the logic stays in one place, `tts/synth.py`.

In [ ]:
%%writefile scripts/synth.py
"""Text -> .wav from a trained experiment.

Example (from the repository root):
    python scripts/synth.py --text "Hello world." --exp expA --out wav_examples/hello.wav

The implementation lives in tts/synth.py, which is written by notebook/A3_final.ipynb (§1.2).
"""
import os, platform, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
if platform.system() == "Darwin" and Path("/opt/homebrew/lib/libespeak-ng.dylib").exists():
    os.environ.setdefault("PHONEMIZER_ESPEAK_LIBRARY", "/opt/homebrew/lib/libespeak-ng.dylib")

from tts.synth import main

if __name__ == "__main__":
    main()

## 1.2.7 Module walk-through: one real batch through every stage (shapes + parameter counts)
This runs an **untrained** model on one real training batch to show how the data flows: text → IDs → encoder → MAS alignment → durations → μ_y → decoder → mel → vocoder → waveform. It also counts parameters per module for each experiment configuration. The output will be noise until the model is trained; the purpose here is to verify the shapes.

**What this cell does:**
1. Reloads every `tts` module, so the code just written is the code that runs.
2. Loads 4 real training clips and pads them into one batch.
3. Runs each training stage separately (encoder → MAS → durations → μ_y → losses) and prints every tensor's shape with its meaning.
4. Checks that MAS gives every frame to exactly one phoneme: the durations must add up to the mel length.
5. Runs inference on one clip and passes the result through the vocoder. Only the shapes matter here.
6. Builds all four configurations and counts the parameters per module.

**Output to expect:**
- two shape tables, training path and inference path;
- one PASS line;
- the parameter table. Only the decoder column changes between experiments.

In [ ]:
# 1.2.7 Module walk-through with real tensors
import tts.align, tts.model, tts.decoders, tts.tts_model, tts.data, tts.train, tts.synth
for m in (tts.align, tts.model, tts.decoders, tts.tts_model, tts.data, tts.train, tts.synth):
    importlib.reload(m)
from tts.data import TTSDataset, collate
from tts.tts_model import TTSModel
from tts.align import log_likelihood, maximum_path

ds = TTSDataset(PROC_DIR / "train_subset.json", MEL_DIR, MEL_STATS)
ids, ids_lens, mels, mel_lens = collate([ds[i] for i in range(4)])
set_seed()
m = TTSModel(len(VOCAB), decoder="flow", dec_blocks=8).eval()
with torch.no_grad():
    h, mu, pad_mask = m.encoder(ids)
    path = maximum_path(log_likelihood(mu, mels), ids_lens, mel_lens)
    durations = path.sum(1)
    mu_y = (path @ mu).transpose(1, 2)
    total, parts = m(ids, ids_lens, mels, mel_lens)
    one = ids[:1, :ids_lens[0]]
    mel_out, d = m.synthesise_mel(one, n_steps=2)
    voc = tts.synth.load_vocoder(VOCODER_PATH, torch.device("cpu"))
    wav = voc(mel_out * MEL_STATS["mel_std"] + MEL_STATS["mel_mean"]).squeeze()

banner("1.2.7 Training path: one batch of 4 real clips")
for name, shape, meaning in [
    ("phoneme ids", tuple(ids.shape), "(batch, max phonemes), padded with 0"),
    ("target mel (normalised)", tuple(mels.shape), "(batch, 80 bands, max frames)"),
    ("encoder states h", tuple(h.shape), "(batch, phonemes, 192)"),
    ("prior mu per phoneme", tuple(mu.shape), "(batch, phonemes, 80)"),
    ("MAS alignment path", tuple(path.shape), "(batch, frames, phonemes), one 1 per frame"),
    ("durations", tuple(durations.shape), f"frames per phoneme; clip 0 sums to {int(durations[0].sum())} = {int(mel_lens[0])} frames"),
    ("mu_y (prior per frame)", tuple(mu_y.shape), "decoder condition"),
]:
    kv(name, shape, meaning)
kv("losses (untrained)", {k: round(v, 3) for k, v in parts.items()}, f"total {total.item():.3f}")
check(bool((durations.sum(1) == mel_lens.float()).all()), "MAS assigns every frame to exactly one phoneme")

banner("1.2.7 Inference path: phonemes → wav (untrained model, shapes only)")
kv("input ids", tuple(one.shape))
kv("predicted durations", tuple(d.shape), f"{int(d.sum())} frames in total")
kv("generated mel", tuple(mel_out.shape))
kv("waveform", tuple(wav.shape), f"{len(wav) / 22050:.2f} s = frames × 256 hop")

banner("1.2.7 Parameters per module for each experiment configuration")
rows = []
for dec in ("flow", "regression"):
    for blocks in (8, 4):
        mm = TTSModel(len(VOCAB), decoder=dec, dec_blocks=blocks)
        count = lambda mod: sum(p.numel() for p in mod.parameters()) / 1e6
        rows.append({"decoder": dec, "blocks": blocks, "encoder (M)": count(mm.encoder), "duration pred. (M)": count(mm.dur),
                     "decoder (M)": count(mm.decoder), "total (M)": count(mm)})
display(pd.DataFrame(rows).round(3))
kv("HiFi-GAN vocoder (frozen)", f"{sum(p.numel() for p in voc.parameters()) / 1e6:.2f} M params", "shared by all experiments")

## 1.2.8 Pipeline wiring
**What this cell does:** imports the single text → wav entry point and prints how to call it from Python and from the terminal, plus the list of modules written above. The real demo with trained checkpoints is in §1.3.6, after training.

In [ ]:
# 1.2.8 Pipeline wiring
# (Optional) Put your pipeline wiring here:
# - load model + vocoder
# - synthesize from a single input text
# - write .wav to wav_examples/
from tts.synth import synthesize_to_wav   # text -> .wav entry point (demonstrated after training, §1.3.6)
banner("1.2.8 Pipeline wiring")
kv("Entry point", "synthesize_to_wav(text, exp, out_path, ckpt='best')")
kv("CLI", "python scripts/synth.py --text \"...\" --exp expA --out wav_examples/x.wav")
kv("Modules written", ", ".join(sorted(p.name for p in TTS_DIR.glob("*.py"))))

# 1.3 Experimental methodology (Rubric 1.3)

## 1.3.1 Controlled experiments (minimum: 2)
Define at least two experiments that share the **same split** and differ by a clearly stated change.

We run a **2 × 2 factorial design**: four experiments, with every pair differing in **exactly one** variable.

| | **8 decoder blocks** | **4 decoder blocks** |
|---|---|---|
| **Flow-matching decoder** | **expA** (baseline) | **expB** |
| **Regression decoder** | **expC** | **expD** |

| Comparison | Controlled change | Question it answers |
|---|---|---|
| expA vs expC, expB vs expD | **decoder objective** (flow matching ↔ regression), same size | Does a *generative* decoder produce more natural mels than plain regression, and what does it cost in speed? |
| expA vs expB, expC vs expD | **decoder depth** (8 ↔ 4 residual blocks), same objective | Does a deeper decoder help, and does the answer depend on the objective? |

**Hypotheses (to be checked against the results):**
- **H1:** regression decoders produce *over-smoothed* mels (the L1 loss predicts the average of all plausible mels), which sound muffled or buzzy. Flow matching produces sharper spectral detail.
- **H2:** regression is faster at inference: 1 decoder pass, against `n_steps` = 10 passes for flow matching.
- **H3:** the 4-block decoders are faster with a small quality loss, and depth matters more for flow matching, which must model a whole distribution.

### Experiment A (baseline)
- **Name:** `expA`
- **Purpose:** the reference system. It is the closest to Matcha-TTS: a flow-matching decoder with 8 residual blocks.
- **Key settings:**
  - Model: Transformer encoder, MAS, duration predictor, and an **OT-CFM flow-matching decoder (8 blocks, 256 channels)**. HiFi-GAN V1 vocoder.
  - Training: full training split (10,480 utterances), **300 epochs**, batch 32, Adam (lr 1e-4), grad-clip 1.0, seed 42.
  - Inference: 10 Euler steps, temperature 0.667.

### Experiment B (controlled change)
- **Name:** `expB`
- **Controlled change vs expA:** decoder depth **8 → 4 blocks**.
- **Key settings:** identical to expA otherwise.

### Experiment C (controlled change)
- **Name:** `expC`
- **Controlled change vs expA:** decoder objective **flow matching → regression** (masked L1 on the mel, one pass, no time conditioning). Same backbone, 8 blocks.
- **Key settings:** identical to expA otherwise.

### Experiment D (controlled change)
- **Name:** `expD`
- **Controlled change:** vs expC, depth 8 → 4. vs expB, objective flow → regression.
- **Key settings:** identical to expA otherwise.

**What is held constant:**
- same split and MD5, same training data (full training split), same preprocessing
- same encoder and duration predictor
- same vocoder
- same seed, optimiser, learning rate, batch size and epochs
- same validation-monitor clips

**Fairness notes:**
- Each experiment's **best checkpoint** is selected by its own validation loss.
- Validation losses **cannot be compared across decoder types**: MSE on velocity is not on the same scale as L1 on the mel. So the comparison *between* experiments uses the objective metrics in §1.4, measured on the same hardware.

*Justification of the outcomes and the key results are added after the runs finish.*

## 1.3.2 What you saved for reproducibility
| Artefact | Path |
|---|---|
| Config files | `configs/expA.json` … `configs/expD.json` (written below) |
| Random seeds | `SEED = 42` in §0.1. Used by the split, the subsets, `set_seed()`, the DataLoader shuffle generator and the fixed validation RNG |
| Split | `results/splits/{train,val,test,train_subset,val_monitor}.txt`, plus `split_md5.txt` |
| Checkpoints | `checkpoints/runs/<exp>/best.ckpt` (lowest validation loss), `last.ckpt` (every epoch, for resuming), `epoch_075/150/225/300.ckpt` (milestones) |
| Logs | `checkpoints/runs/<exp>/train_log.csv` (per-epoch train/validation losses, components, seconds) |
| Preprocessing artefacts | `results/vocab.json`, `results/mel_stats.json`, `results/preprocessing_stats.csv` |

---
### Code in §1.3, in run order
| # | Cell | What it does | Main outputs |
|---|---|---|---|
| 1.3.3 | Experiment configs | one shared base + one change per experiment; proves each pair differs in one key | `configs/expA–D.json` |
| 1.3.4 | Training | trains expA → expD with the same `train()` | checkpoints, `train_log.csv` |
| 1.3.5 | Training summary | best epochs, times, curves | `training_summary.csv`, figure |
| 1.3.6 | Inference demo | text → wav with every experiment, plus the CLI | `results/demo/*.wav` |

## 1.3.3 Experiment configs
**What this cell does:**
1. `BASE` holds every shared setting: seed, split fingerprint, subset size, epochs, batch size, learning rate, gradient clip, milestone epochs, and the inference settings.
2. `EXPERIMENTS` lists each experiment as **only its change** from `BASE`, so the controlled variable is visible in one line.
3. Writes the merged configs to `configs/expA.json` … `configs/expD.json`. These files are what `train()` and the CLI read.
4. `load_config()` adds this machine's paths (processed data, mels, checkpoint folder) to a config file.
5. `diff_configs()` lists the keys that differ between two experiments. The cell **asserts** that each controlled pair differs in exactly one key.

**Output to expect:** the grid table, the four pairwise differences (each marked "exactly one variable"), and the milestone epochs.

In [ ]:
# 1.3.3 Experiment configs: one base config + exactly one change per experiment, written to configs/
BASE = {"seed": SEED, "split_md5": SPLIT_MD5, "n_train_utts": len(SUBSETS["train_subset"]), "epochs": EPOCHS,
        "batch_size": 32 if not SMOKE else 16, "lr": 1e-4, "grad_clip": 1.0,
        "milestones": sorted({max(1, round(EPOCHS * f)) for f in (0.25, 0.5, 0.75, 1.0)}),
        "decoder": "flow", "dec_blocks": 8, "inference_steps": 10, "temperature": 0.667}
EXPERIMENTS = {
    "expA": {},                                            # baseline: flow matching, 8 blocks
    "expB": {"dec_blocks": 4},                             # depth
    "expC": {"decoder": "regression"},                     # objective
    "expD": {"decoder": "regression", "dec_blocks": 4},    # objective + depth (vs expB: objective, vs expC: depth)
}
CONFIGS = {}
for name, change in EXPERIMENTS.items():
    CONFIGS[name] = {"name": name, **BASE, **change}                   # later keys win: the change overrides BASE
    (CONFIG_DIR / f"{name}.json").write_text(json.dumps(CONFIGS[name], indent=2))

def load_config(name):
    """Config file + machine-specific paths."""
    cfg = json.loads((CONFIG_DIR / f"{name}.json").read_text())
    return {**cfg, "proc_dir": str(PROC_DIR), "mel_dir": str(MEL_DIR), "ckpt_dir": str(CKPT_ROOT / name)}

def diff_configs(a, b):
    """{key: (value in a, value in b)} for every key that differs, ignoring the name."""
    return {k: (CONFIGS[a][k], CONFIGS[b][k]) for k in CONFIGS[a] if k != "name" and CONFIGS[a][k] != CONFIGS[b][k]}

banner("1.3.3 Experiment grid")
display(pd.DataFrame(CONFIGS).T[["decoder", "dec_blocks", "n_train_utts", "epochs", "batch_size", "lr", "seed", "split_md5"]].astype(str))
banner("1.3.3 Controlled pairs: keys that differ (everything else identical)")
for a, b in [("expA", "expC"), ("expB", "expD"), ("expA", "expB"), ("expC", "expD")]:
    d = diff_configs(a, b)
    kv(f"{a} vs {b}", d, "exactly one variable" if len(d) == 1 else "MORE THAN ONE VARIABLE")
    assert len(d) == 1
kv("Milestone checkpoints at epochs", BASE["milestones"])
kv("Configs written", ", ".join(f"configs/{n}.json" for n in CONFIGS))

## 1.3.4 Train the four experiments
**What this cell does:** for each experiment in `RUN`, it prints the equivalent terminal command and calls the **same** `train()` from §1.2.5.6 with that experiment's config. For each experiment it prints:
- one header line: decoder, blocks, parameter count, steps per epoch, device;
- one progress line every 10 epochs: train loss, validation loss with its three parts (dur / prior / dec), and seconds per epoch. `★ best` means `best.ckpt` was updated, and `◆ milestone saved` means a milestone checkpoint was written;
- a final line with the best validation loss, its epoch, and the total training time.

**Safe to re-run:** each experiment resumes from its `last.ckpt` (for example after a Colab disconnect or a laptop sleep) and is skipped once it has reached `EPOCHS`. To train only some experiments, edit `RUN_EXPERIMENTS` in §0.1.

**Time:** about 27 s per epoch on a Colab T4 (327 steps), so about 2.2 hours per experiment for 300 epochs. The measured times for all four runs are in §1.3.5.

In [ ]:
# 1.3.4 Train the four experiments
# (Optional) Run/record your experiment commands here (or print them):
# !python scripts/train.py --config configs/expA.yaml
# !python scripts/train.py --config configs/expB.yaml
#
# Train all four experiments. Safe to re-run: each run resumes from its last.ckpt (e.g. after a Colab disconnect)
# and is skipped once it has reached EPOCHS.
from tts.train import train
RUN = RUN_EXPERIMENTS                           # set in §0.1, e.g. ["expC"]
LOGS = {}
for name in RUN:
    banner(f"Training {name}: {CONFIGS[name]['decoder']} decoder, {CONFIGS[name]['dec_blocks']} blocks")
    print(f"  equivalent CLI: python -m tts.train --config configs/{name}.json --proc_dir {PROC_DIR} --mel_dir {MEL_DIR} --ckpt_root {CKPT_ROOT}")
    LOGS[name] = train(load_config(name), verbose_every=1 if SMOKE else 10)

## 1.3.5 Training summary and curves
**What this cell does:**
1. Reads every experiment's `train_log.csv`, so it also works in a fresh session without retraining.
2. Builds one row per experiment:
   - best epoch and best validation loss, with its three parts;
   - seconds per epoch and total training time;
   - the checkpoint files saved.
3. Saves the table to `results/training_summary.csv`.
4. Plots four curves (`1_3_training_curves.png`): train total loss, validation total loss (★ = the epoch kept as `best.ckpt`), validation duration loss and validation prior loss. Solid lines are flow matching, dashed lines regression.

**How to read it:**
- The **duration** and **prior** losses come from identical modules, so they *can* be compared across all four runs.
- The **total** and **decoder** losses use different objectives (velocity MSE vs mel L1), so compare them only *within* a decoder type.
- Quality *between* experiments is judged with the §1.4 metrics.

In [ ]:
# 1.3.5 Training summary + curves (read from each train_log.csv, so this works after a restart too)
logs = {n: pd.read_csv(CKPT_ROOT / n / "train_log.csv") for n in CONFIGS if (CKPT_ROOT / n / "train_log.csv").exists()}
rows = []
for n, lg in logs.items():
    b = lg.loc[lg.val.idxmin()]                                          # the epoch saved as best.ckpt
    ckpts = sorted(p.name for p in (CKPT_ROOT / n).glob("*.ckpt"))
    rows.append({"exp": n, "decoder": CONFIGS[n]["decoder"], "blocks": CONFIGS[n]["dec_blocks"], "epochs run": int(lg.epoch.max()),
                 "best epoch": int(b.epoch), "best val": b.val, "val dur @best": b.val_dur, "val prior @best": b.val_prior,
                 "val dec @best": b.val_dec, "s/epoch": lg.sec.mean(), "train time (min)": lg.sec.sum() / 60,
                 "checkpoints": ", ".join(ckpts)})
banner("1.3.5 Training summary")
summary_runs = pd.DataFrame(rows).set_index("exp")
display(summary_runs.round(4))
summary_runs.to_csv(RESULTS_DIR / "training_summary.csv")
print("  saved → results/training_summary.csv")
print("  note: 'val dec' is MSE on flow velocity (flow) vs L1 on the mel (regression), so it is NOT comparable across")
print("        decoder types. 'val dur' and 'val prior' come from identical modules and ARE comparable.")

fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))
for n, lg in logs.items():
    style = "-" if CONFIGS[n]["decoder"] == "flow" else "--"
    ax[0].plot(lg.epoch, lg.train, style, label=n); ax[1].plot(lg.epoch, lg.val, style, label=n)
    ax[2].plot(lg.epoch, lg.val_dur, style, label=n); ax[3].plot(lg.epoch, lg.val_prior, style, label=n)
    ax[1].scatter(lg.loc[lg.val.idxmin(), "epoch"], lg.val.min(), marker="*", s=120)
for a, title in zip(ax, ["train total loss", "val total loss (★ = best.ckpt)", "val duration loss", "val prior loss"]):
    a.set(title=title, xlabel="epoch"); a.legend(fontsize=8)
fig.suptitle("Training curves (solid = flow matching, dashed = regression)", fontsize=12)
plt.tight_layout(); savefig("1_3_training_curves"); plt.show()

## 1.3.6 Inference demo (template 1.2.4)
**What this cell does:**
1. Synthesises one LJSpeech-style sentence with each experiment's `best.ckpt` through `synthesize_to_wav()`.
2. Saves each result to `results/demo/demo_<exp>.wav` and prints the audio length and the synthesis time, with an audio player.
3. Runs the command-line script `scripts/synth.py` once in a separate process. This proves the terminal entry point works on its own.

**Note:** the first call per experiment includes loading the model, so these times are *not* the RTF or latency figures. Those are measured properly, with warm-up and repeats, in §1.4.

**Output to expect:** one line and one player per experiment, then the CLI output (the normalised text and the wav that was written).

In [ ]:
# 1.3.6 (template 1.2.4) demonstration: text -> .wav with every experiment's best checkpoint, plus the CLI entry point
import tts.synth; importlib.reload(tts.synth)
from tts.synth import synthesize_to_wav
DEMO_DIR = RESULTS_DIR / "demo"
demo_text = "Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts."
banner("1.3.6 Inference demo (best checkpoints)")
print(f"  text: {demo_text}\n")
for n in logs:
    t0 = time.perf_counter()
    wav, sr = synthesize_to_wav(demo_text, exp=n, out_path=DEMO_DIR / f"demo_{n}.wav", ckpt_root=CKPT_ROOT, vocoder_path=VOCODER_PATH)
    kv(n, f"{len(wav) / sr:.2f} s audio in {time.perf_counter() - t0:.2f} s", f"results/demo/demo_{n}.wav")
    display(Audio(wav, rate=sr))

cmd = [sys.executable, "scripts/synth.py", "--text", "Hello world, this is our text to speech system.", "--exp", "expA",
       "--out", str(DEMO_DIR / "cli_test.wav"), "--ckpt_root", str(CKPT_ROOT), "--vocoder", str(VOCODER_PATH)]
print("\n  CLI:", " ".join(cmd[1:]))
print("  " + subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.replace("\n", "\n  "))

# 1.4 Evaluation, coding practice & attribution (Rubric 1.4)

## 1.4.1 Objective metric (minimum: 1) — report for each experiment
Implement and report at least one objective metric for **each** experiment, e.g.:
- **Latency:** average synthesis time per sentence (state number of runs)
- **Real-Time Factor (RTF):** synthesis_time / audio_duration
- **Other justified metric:** `<describe and justify>`

Report results in a small table (in this notebook)

**Our metrics.** Every experiment is scored with all of them. Each answers a different question:

| Metric | What it measures | How | Better | Why we use it |
|---|---|---|---|---|
| **WER / CER** | intelligibility | Whisper `small.en` (Radford et al., 2022) transcribes each clip; word / character error rate against the text the model was asked to say. Both texts go through our normaliser, so "1963" = "nineteen sixty-three" | ↓ | An automatic listener: if a word is slurred, skipped or mispronounced, Whisper gets it wrong |
| **MCD** (dB) | spectral distance to the real recording | WORLD envelope → SPTK mel-cepstrum (order 24, α = 0.455, c0 dropped), aligned with DTW (Kubichek, 1993) | ↓ | Catches muffled / wrong-timbre speech. *Caveat:* it rewards the average spectrum, so it favours regression over sampled detail (§1.4.1.7 shows this) |
| **Duration ratio** | speaking rate | synthesised length / recorded length | → 1 | Tests the duration predictor (rhythm) |
| **RTF**, **latency** | speed | end-to-end text → wav, and per stage (front-end, acoustic model, vocoder); medians after 2 warm-ups × 3 repeats × 12 prompts, on MPS and on CPU | ↓ | The cost side of the trade-off; RTF < 1 = faster than real time |
| **Validation loss** | checkpoint selection | duration + prior + decoder loss on all 1,310 validation clips | ↓ | Confirms that the 300-clip monitor used to pick `best.ckpt` agrees with the full validation split |

**Systems scored on the test split:** the real recording (upper bound), the *vocoded* real recording (real mel → HiFi-GAN: the best our pipeline could do with a perfect acoustic model), expA–D trained on the full split (main results), and expA–D trained on the 3,000-utterance subset (data-size comparison, RQ3). The test split was never used for training or checkpoint selection.

**Code in §1.4, in run order:**

| Cell | Purpose | Writes |
|---|---|---|
| 1.4.1.1 | evaluation library | `tts/evaluate.py` |
| 1.4.1.2 | command-line entry point | `scripts/evaluate.py` |
| 1.4.1.3 | settings + MCD sanity check | – |
| 1.4.1.4 | test-set scores per experiment | `results/eval/per_utterance.csv`, `summary.csv`, `metrics.json` |
| 1.4.1.5 | controlled comparisons with confidence intervals | `results/eval/comparisons.csv` |
| 1.4.1.6 | speed: RTF and latency | `results/eval/speed.csv` |
| 1.4.1.7 | flow-matching step sweep | `results/eval/step_sweep.csv` |
| 1.4.1.8 | validation re-score of every checkpoint | `results/eval/val_rescore.csv` |
| 1.4.2.1 | ≥10 prompts → `.wav` for every experiment | `wav_examples/` |
| 1.4.2.2 | spectrogram comparison | `results/spectrograms/` |
| Final | artefact check | – |

### 1.4.1.1 Evaluation library (`tts/evaluate.py`)
**What this cell does:** writes the evaluation code to `tts/evaluate.py` with `%%writefile`, like the model modules in §1.2. Nothing runs yet. It defines:
1. `test_sentences()`: a seeded sample of the test split (`n=0` = all 1,310 clips) with the normalised transcripts.
2. `mel_cepstrum()`, `mcd_dtw()`: the standard MCD. A first version computed cepstra from our own 80-band log-mels; the check in §1.4.1.3 showed that the log floor of near-silent bands dominated it, so it was replaced with WORLD/SPTK features.
3. `text_for_wer()`, `Whisper`: the automatic listener and the shared text normalisation.
4. `make_systems()`, `score_test_set()`: synthesises every test sentence with every system and scores it. Results are saved per utterance after each system, and pairs already in the CSV are skipped, so an interrupted run resumes. The noise seed is the sentence index, so all flow models get the same noise for the same sentence.
5. `summarise()`, `paired_difference()`: corpus WER/CER, mean MCD and duration ratio with **95% bootstrap confidence intervals** over sentences, and a *paired* bootstrap for differences between two systems.
6. `measure_speed()`, `summarise_speed()`: RTF and latency, timed per stage.
7. `step_sweep()`: flow-matching quality and speed at 1–20 Euler steps.
8. `rescore_validation()`: validation loss of every saved checkpoint on all validation clips and on the 300-clip monitor.

**Output to expect:** `Writing tts/evaluate.py` (or `Overwriting` on a re-run).

In [ ]:
%%writefile tts/evaluate.py
"""Objective evaluation (notebook §1.4): intelligibility, spectral distance, rhythm, speed, step sweep, validation re-score.

Metrics
  - WER / CER: Whisper small.en (Radford et al., 2022) transcribes each clip; reference and transcript go through the
    same text normaliser (tts.text.normalize_text), so "1963" and "nineteen sixty-three" match.
  - MCD: mel-cepstral distortion in dB (Kubichek, 1993) on WORLD/SPTK mel-cepstra (order 24, alpha 0.455, c0 dropped),
    averaged over the DTW alignment of synthesised and recorded speech.
  - duration ratio: synthesised length / recorded length (1.0 = the speaker's own pace).
  - RTF and latency: end-to-end and per stage (front-end, acoustic model, vocoder), medians after warm-up.
All test-set scores are saved per utterance, so a run can be resumed and any table can be rebuilt from the CSV.
"""
import csv, json, time
from pathlib import Path

import jiwer, librosa, numpy as np, pandas as pd, soundfile as sf, torch
from torch.utils.data import DataLoader

from tts.audio import load_wav, mel_spectrogram
from tts.data import TTSDataset, collate
from tts.synth import load_tts, load_vocoder, synthesize, SR
from tts.text import normalize_text, phonemes_to_ids, text_to_ids
from tts.train import evaluate as val_losses

EXPS = ["expA", "expB", "expC", "expD"]


# ---------------------------------------------------------------- data
def load_metadata(data_dir):
    """LJSpeech metadata; QUOTE_NONE so quote characters are not treated as CSV quoting (see §1.1.6)."""
    return pd.read_csv(Path(data_dir) / "metadata.csv", sep="|", header=None, quoting=csv.QUOTE_NONE,
                       names=["id", "raw", "norm"], dtype=str).set_index("id")


def test_sentences(split_dir, data_dir, n=0, seed=42):
    """A seeded sample of n test clips (n=0: the whole test split) as [(id, normalised transcript)]."""
    meta = load_metadata(data_dir)
    ids = (Path(split_dir) / "test.txt").read_text().split()
    ids = sorted(np.random.default_rng(seed).permutation(sorted(ids))[:n or None])
    return [(i, meta.loc[i, "norm"]) for i in ids]


# ---------------------------------------------------------------- metrics
def mel_cepstrum(wav, order=24, alpha=0.455):
    """WORLD spectral envelope -> SPTK mel-cepstrum (alpha 0.455 suits 22.05 kHz); returns (frames, order), c0 dropped."""
    import pysptk, pyworld
    x = wav.astype(np.float64)
    f0, t = pyworld.dio(x, SR, frame_period=5.0)                       # F0 is only needed to estimate the envelope
    f0 = pyworld.stonemask(x, f0, t, SR)
    sp = pyworld.cheaptrick(x, f0, t, SR, fft_size=1024)
    return pysptk.sp2mc(sp, order=order, alpha=alpha)[:, 1:]


def mcd_dtw(ref, syn):
    """Mel-cepstral distortion (dB) over the DTW alignment: the two utterances have different timing."""
    a, b = mel_cepstrum(ref), mel_cepstrum(syn)
    _, path = librosa.sequence.dtw(a.T, b.T, metric="euclidean")
    d = a[path[:, 0]] - b[path[:, 1]]
    return float(10 / np.log(10) * np.sqrt(2) * np.mean(np.sqrt((d ** 2).sum(1))))


def text_for_wer(s):
    """Normalise (numbers spelled out, lower case), then keep only letters, apostrophes and spaces."""
    s = normalize_text(s)
    return " ".join("".join(c if c.isalpha() or c in " '" else " " for c in s).split())


class Whisper:
    """Automatic listener for WER/CER."""
    def __init__(self, device, name="openai/whisper-small.en"):
        from transformers import pipeline
        self.asr = pipeline("automatic-speech-recognition", model=name, device=device)

    def __call__(self, wav, sr=SR):
        return self.asr({"raw": librosa.resample(wav, orig_sr=sr, target_sr=16000), "sampling_rate": 16000})["text"].strip()


# ---------------------------------------------------------------- systems
def make_systems(device, vocoder, runs):
    """name -> fn(text, recorded wav) -> wav. runs = {"full": ckpt_root, "subset": ckpt_root}; missing roots are skipped."""
    out = {"ground_truth": lambda text, gt: gt,
           "vocoded_gt": lambda text, gt: vocoder(mel_spectrogram(gt)[None].to(device)).squeeze().clamp(-1, 1).cpu().numpy()}
    for run, root in runs.items():
        for e in EXPS:
            path = Path(root) / e / "best.ckpt"
            if path.exists():
                model, ck = load_tts(path, device)
                out[f"{run}_{e}"] = (lambda m, c: lambda text, gt: synthesize(text, m, vocoder, c["vocab"], c["mel_stats"])[0])(model, ck)
    return out


# ---------------------------------------------------------------- test-set quality
@torch.no_grad()
def score_test_set(sents, runs, data_dir, vocoder_path, out_dir, device, save_wavs=5):
    """Scores every (system, sentence) pair not already in out_dir/per_utterance.csv, saving after each system.
    The noise seed is the sentence index, so every flow model hears the same noise for the same sentence."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "per_utterance.csv"
    done = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame(columns=["system", "id"])
    vocoder = load_vocoder(vocoder_path, device)
    systems, asr = make_systems(device, vocoder, runs), None
    for name, fn in systems.items():
        have = set(done.id[done.system == name])
        todo = [(k, cid, text) for k, (cid, text) in enumerate(sents) if cid not in have]
        if not todo:
            print(f"  {name:14s} all {len(sents)} sentences already scored"); continue
        asr = asr or Whisper(device)
        rows, t0 = [], time.time()
        for k, cid, text in todo:
            torch.manual_seed(k)
            gt = load_wav(Path(data_dir) / "wavs" / f"{cid}.wav")                  # trimmed, exactly as in training
            wav = fn(text, gt)
            hyp = asr(wav)
            r, h = text_for_wer(text), text_for_wer(hyp)
            rows.append({"system": name, "id": cid, "text": text, "whisper": hyp, "wer": jiwer.wer(r, h),
                         "cer": jiwer.cer(r, h), "mcd": mcd_dtw(gt, wav), "dur_ratio": len(wav) / len(gt), "ref_words": len(r.split())})
            if k < save_wavs:
                (out_dir / "test_wavs").mkdir(exist_ok=True)
                sf.write(out_dir / "test_wavs" / f"{name}_{cid}.wav", wav, SR)
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(csv_path, index=False)
        print(f"  {name:14s} {len(todo)} sentences scored in {time.time() - t0:.0f} s", flush=True)
    ids = {cid for cid, _ in sents}
    return done[done.id.isin(ids) & done.system.isin(systems)].reset_index(drop=True)


def corpus_wer(g, metric=jiwer.wer):
    """Corpus-level error rate (%): errors summed over all sentences, not an average of per-sentence rates."""
    return 100 * metric([text_for_wer(t) for t in g.text], [text_for_wer(h) for h in g.whisper.fillna("")])


def _boot(g, stat, n_boot, seed):
    rng = np.random.default_rng(seed)
    return np.array([stat(g.iloc[rng.integers(0, len(g), len(g))]) for _ in range(n_boot)])


def summarise(df, n_boot=1000, seed=0):
    """One row per system: WER, CER, MCD, duration ratio, each with a 95% bootstrap CI over sentences."""
    stats = {"WER_%": corpus_wer, "CER_%": lambda g: corpus_wer(g, jiwer.cer),
             "MCD_dB": lambda g: g.mcd.mean(), "dur_ratio": lambda g: g.dur_ratio.mean()}
    rows = []
    for name, g in df.groupby("system", sort=False):
        g = g.sort_values("id").reset_index(drop=True)
        row = {"system": name, "n": len(g)}
        for k, f in stats.items():
            lo, hi = np.percentile(_boot(g, f, n_boot, seed), [2.5, 97.5])
            row.update({k: f(g), f"{k}_lo": lo, f"{k}_hi": hi})
        rows.append(row)
    return pd.DataFrame(rows).set_index("system")


def paired_difference(df, a, b, n_boot=1000, seed=0):
    """b minus a for WER and MCD, with a paired bootstrap CI (the same sentences are resampled for both systems).
    If the CI excludes 0, the difference is unlikely to be an accident of which sentences were sampled."""
    ga = df[df.system == a].sort_values("id").reset_index(drop=True)
    gb = df[df.system == b].sort_values("id").reset_index(drop=True)
    assert list(ga.id) == list(gb.id), "systems were scored on different sentences"
    rng, out = np.random.default_rng(seed), {}
    for k, f in {"WER_%": corpus_wer, "MCD_dB": lambda g: g.mcd.mean()}.items():
        diffs = []
        for _ in range(n_boot):
            i = rng.integers(0, len(ga), len(ga))
            diffs.append(f(gb.iloc[i]) - f(ga.iloc[i]))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        out.update({f"Δ{k}": f(gb) - f(ga), f"Δ{k}_lo": lo, f"Δ{k}_hi": hi})
    return out


# ---------------------------------------------------------------- speed
@torch.no_grad()
def measure_speed(prompts, ckpt_root, vocoder_path, device, n_warm=2, n_rep=3):
    """End-to-end and per-stage timing on the given prompts (warm-up runs are discarded)."""
    sync = {"mps": torch.mps.synchronize, "cuda": torch.cuda.synchronize}.get(device.type, lambda: None)
    vocoder, rows = load_vocoder(vocoder_path, device), []
    for e in EXPS:
        model, ck = load_tts(Path(ckpt_root) / e / "best.ckpt", device)
        st = ck["mel_stats"]

        def run(text):
            """The steps of tts.synth.synthesize, timed separately."""
            t0 = time.perf_counter()
            ids = torch.tensor([text_to_ids(text, ck["vocab"])], device=device)          # normaliser + phonemiser
            t1 = time.perf_counter()
            mel, _ = model.synthesise_mel(ids, n_steps=10, temperature=0.667); sync()     # acoustic model
            t2 = time.perf_counter()
            wav = vocoder(mel * st["mel_std"] + st["mel_mean"]).squeeze().cpu().numpy()  # vocoder (.cpu() waits for it)
            t3 = time.perf_counter()
            return wav, (t1 - t0, t2 - t1, t3 - t2, t3 - t0)

        for t in prompts[:n_warm]:
            run(t)
        for t in prompts:
            for _ in range(n_rep):
                wav, (tf, ta, tv, tt) = run(t)
                a = len(wav) / SR
                rows.append({"device": device.type, "exp": e, "audio_s": a, "latency_s": tt, "frontend_s": tf,
                             "acoustic_s": ta, "vocoder_s": tv, "rtf": tt / a, "rtf_acoustic": ta / a})
    return pd.DataFrame(rows)


def summarise_speed(raw):
    return raw.groupby(["device", "exp"]).agg(
        RTF=("rtf", "median"), RTF_acoustic=("rtf_acoustic", "median"), latency_s=("latency_s", "median"),
        latency_p90_s=("latency_s", lambda x: x.quantile(0.9)), frontend_s=("frontend_s", "median"),
        acoustic_s=("acoustic_s", "median"), vocoder_s=("vocoder_s", "median"), runs=("rtf", "size"))


# ---------------------------------------------------------------- flow-matching step sweep
@torch.no_grad()
def step_sweep(sents, ckpt_path, data_dir, vocoder_path, device, steps=(1, 2, 4, 10, 20)):
    """Number of Euler steps at inference: the quality/speed knob that only flow matching has."""
    vocoder = load_vocoder(vocoder_path, device)
    model, ck = load_tts(ckpt_path, device)
    asr, rows = Whisper(device), []
    for n in steps:
        for k, (cid, text) in enumerate(sents):
            torch.manual_seed(k)
            gt = load_wav(Path(data_dir) / "wavs" / f"{cid}.wav")
            t0 = time.perf_counter()
            wav = synthesize(text, model, vocoder, ck["vocab"], ck["mel_stats"], n_steps=n)[0]
            dt = time.perf_counter() - t0
            rows.append({"steps": n, "id": cid, "text": text, "whisper": asr(wav), "mcd": mcd_dtw(gt, wav), "rtf": dt / (len(wav) / SR)})
    df = pd.DataFrame(rows)
    summary = pd.DataFrame([{"steps": n, "WER_%": corpus_wer(g), "MCD_dB": g.mcd.mean(), "RTF": g.rtf.median()}
                            for n, g in df.groupby("steps")]).set_index("steps")
    return summary, df


# ---------------------------------------------------------------- validation re-score
def _records(ids, phonemes, vocab, mel_dir, data_dir):
    """Training-format records for any clips; computes (and caches) the trimmed log-mel of clips not cached yet."""
    recs = []
    for cid in ids:
        p = Path(mel_dir) / f"{cid}.pt"
        if not p.exists():
            torch.save(mel_spectrogram(load_wav(Path(data_dir) / "wavs" / f"{cid}.wav")).half(), p)
        recs.append({"id": cid, "phonemes": phonemes_to_ids(phonemes[cid], vocab), "frames": int(torch.load(p).shape[1])})
    return recs


@torch.no_grad()
def rescore_validation(ckpt_root, val_ids, monitor_ids, phonemes, mel_dir, data_dir, out_dir, device, seed=42, batch=32):
    """Validation loss of every saved checkpoint on ALL validation clips and on the 300-clip monitor set.
    Checks that the monitor set used to pick best.ckpt agrees with the full validation split."""
    rows, tmp = [], Path(out_dir) / "_val_records.json"
    for e in EXPS:
        for path in sorted((Path(ckpt_root) / e).glob("*.ckpt")):
            if path.name == "last.ckpt":
                continue                                                       # identical to epoch_300.ckpt
            model, ck = load_tts(path, device)
            row = {"exp": e, "checkpoint": path.stem, "epoch": ck["epoch"]}
            for name, ids in (("all_val", val_ids), ("monitor", monitor_ids)):
                tmp.write_text(json.dumps(_records(ids, phonemes, ck["vocab"], mel_dir, data_dir)))
                dl = DataLoader(TTSDataset(tmp, mel_dir, ck["mel_stats"]), batch_size=batch, collate_fn=collate)
                parts = val_losses(model, dl, device, seed)
                row.update({f"{name}_total": sum(parts.values()), f"{name}_dur": parts["dur"]})
            rows.append(row)
    tmp.unlink(missing_ok=True)
    return pd.DataFrame(rows)

### 1.4.1.2 Command-line evaluation (`scripts/evaluate.py`)
**What this cell does:** writes the terminal entry point the template asks for (`scripts/evaluate.py`, "objective metric computation"). It scores the test split and writes `results/eval/summary.csv` and `metrics.json`; the cells below call the same functions directly.

**Output to expect:** `Writing scripts/evaluate.py`.

In [ ]:
%%writefile scripts/evaluate.py
"""Objective evaluation from the command line (the library is tts/evaluate.py, written by the notebook, §1.4).

Example (from the repository root):
    python scripts/evaluate.py --n_test 0        # whole test split; resumes if interrupted
Writes results/eval/per_utterance.csv, summary.csv and metrics.json.
"""
import argparse, json, os, platform, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
if platform.system() == "Darwin" and Path("/opt/homebrew/lib/libespeak-ng.dylib").exists():
    os.environ.setdefault("PHONEMIZER_ESPEAK_LIBRARY", "/opt/homebrew/lib/libespeak-ng.dylib")

from tts.evaluate import score_test_set, summarise, test_sentences
from tts.train import pick_device

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_test", type=int, default=0, help="test sentences (0 = the whole test split)")
    ap.add_argument("--full_root", default="checkpoints/runs_full")
    ap.add_argument("--subset_root", default="checkpoints/runs")
    ap.add_argument("--data_dir", default="data/LJSpeech-1.1")
    ap.add_argument("--split_dir", default="results/splits")
    ap.add_argument("--vocoder", default="checkpoints/hifigan_lj_v1")
    ap.add_argument("--out_dir", default="results/eval")
    a = ap.parse_args()
    sents = test_sentences(a.split_dir, a.data_dir, a.n_test)
    df = score_test_set(sents, {"full": a.full_root, "subset": a.subset_root}, a.data_dir, a.vocoder, a.out_dir, pick_device())
    s = summarise(df)
    s.to_csv(Path(a.out_dir) / "summary.csv")
    (Path(a.out_dir) / "metrics.json").write_text(json.dumps(s.round(4).reset_index().to_dict("records"), indent=1))
    print(s.round(3).to_string())

### 1.4.1.3 Evaluation settings and MCD sanity check
**What this cell does:**
1. Loads the library, picks the test sentences (`N_TEST = 0`: the whole test split, 1,310 clips; 8 in smoke mode) and the 12 listening prompts.
2. Finds the checkpoints: the main runs in `checkpoints/runs_full/` and, if present, the 3,000-utterance runs in `checkpoints/runs/` (trained by `A3_final.ipynb`).
3. **Checks that MCD behaves sensibly** on cases with a known answer: a recording against itself time-stretched by 8% (should be small), against itself with faint noise, and against a *different* sentence (should be large).

**Output to expect:** the settings, then three MCD values that increase from top to bottom.

In [ ]:
# 1.4.1.3 Evaluation settings + MCD sanity check
import tts.evaluate as EV; importlib.reload(EV)
EVAL_DIR = RESULTS_DIR / "eval"; EVAL_DIR.mkdir(parents=True, exist_ok=True)
N_TEST = 8 if SMOKE else 0                                 # 0 = the whole test split
RUNS = {"full": CKPT_ROOT}                                 # main runs: full training split
SUBSET_ROOT = PROJECT_ROOT / "checkpoints" / "runs"        # 3,000-utterance runs (A3_final.ipynb), for RQ3
if (SUBSET_ROOT / "expA" / "best.ckpt").exists() and not SMOKE:
    RUNS["subset"] = SUBSET_ROOT
TEST = EV.test_sentences(SPLIT_DIR, DATA_DIR, N_TEST)
PROMPTS = [
    "The quick brown fox jumps over the lazy dog.",
    "Hello, welcome to our text to speech experiment.",
    "On March 3rd, 1963, Mr. Johnson paid $3.50 for a ticket at 10:30.",
    "Do you really think that the committee will approve the new proposal by Friday?",
    "The old library, which had been closed for nearly a decade, finally reopened its doors to the public last spring.",
    "Dr. Smith lives at 221 Baker St. in London.",
    "Artificial intelligence can transform written text into speech.",
    "She sells seashells by the seashore.",
    "The commission concluded that the evidence did not support the theory of a second gunman.",
    "Wow! That was absolutely incredible, wasn't it?",
    "Please read this sentence clearly and naturally.",
    "Although the weather forecast predicted heavy rain throughout the afternoon, thousands of people still gathered in the park to watch the annual music festival.",
]

banner("1.4.1.3 Evaluation settings")
kv("Test sentences", f"{len(TEST):,}", "whole test split" if not N_TEST else "sample")
kv("Listening prompts", len(PROMPTS), "not from LJSpeech: generalisation to new text")
for run, root in RUNS.items():
    kv(f"Checkpoints ({run})", root.relative_to(PROJECT_ROOT), ", ".join(e for e in EV.EXPS if (root / e / "best.ckpt").exists()))
kv("Device", DEVICE)

banner("MCD sanity check (known answers)")
a = A.load_wav(DATA_DIR / "wavs" / f"{TEST[0][0]}.wav"); b = A.load_wav(DATA_DIR / "wavs" / f"{TEST[1][0]}.wav")
noise = np.random.default_rng(0).standard_normal(len(a)).astype(np.float32) * a.std() * 10 ** (-30 / 20)
kv("same recording, 8% slower", f"{EV.mcd_dtw(a, librosa.effects.time_stretch(a, rate=1 / 1.08)):.2f} dB", "small: DTW absorbs the timing")
kv("same recording + noise at 30 dB SNR", f"{EV.mcd_dtw(a, a + noise):.2f} dB", "MCD is sensitive to fine spectral texture")
kv("a different sentence", f"{EV.mcd_dtw(a, b):.2f} dB", "large: different sounds")

### 1.4.1.4 Test-set results per experiment
**What this cell does:**
1. Scores every system on the test sentences (`score_test_set`). Pairs already in `results/eval/per_utterance.csv` are loaded instead of recomputed; the whole test split takes several hours on a laptop the first time (it was run once with `python scripts/evaluate.py`).
2. Summarises each system: WER, CER, MCD and duration ratio, each with a 95% bootstrap confidence interval (1,000 resamples of the sentences).
3. Saves `results/eval/summary.csv` and `metrics.json` (the template's metrics file), and prints the table for the four main experiments, then all systems.

**How to read it:** the ground-truth row is Whisper's own error rate on real speech, the floor for WER. *Vocoded GT* shows how much the vocoder alone costs. Two systems differ reliably only if their intervals do not overlap much; §1.4.1.5 tests the differences directly.

In [ ]:
# 1.4.1.4 Test-set quality per experiment (cached per utterance; resumes if interrupted)
banner("1.4.1.4 Test-set scoring")
per_utt = EV.score_test_set(TEST, RUNS, DATA_DIR, VOCODER_PATH, EVAL_DIR, DEVICE)
summary = EV.summarise(per_utt)
summary.to_csv(EVAL_DIR / "summary.csv")
(EVAL_DIR / "metrics.json").write_text(json.dumps(summary.round(4).reset_index().to_dict("records"), indent=1))

def with_ci(s, k, nd=2):
    return s.apply(lambda r: f"{r[k]:.{nd}f} [{r[k + '_lo']:.{nd}f}, {r[k + '_hi']:.{nd}f}]", axis=1)

def table(s):
    return pd.DataFrame({"n": s.n, "WER % [95% CI]": with_ci(s, "WER_%"), "CER % [95% CI]": with_ci(s, "CER_%"),
                         "MCD dB [95% CI]": with_ci(s, "MCD_dB"), "duration ratio": s.dur_ratio.round(3)})

main = [f"full_{e}" for e in EV.EXPS]
banner("Main results: the four experiments (full training split), test set")
display(table(summary.loc[main]).rename(index=lambda n: n.replace("full_", "") + f" ({CONFIGS[n[5:]]['decoder']}, {CONFIGS[n[5:]]['dec_blocks']} blocks)"))
banner("All systems")
display(table(summary))
print("  saved → results/eval/summary.csv, results/eval/metrics.json, results/eval/per_utterance.csv")

### 1.4.1.5 Controlled comparisons (RQ1–RQ3)
**What this cell does:** for each controlled pair, computes the difference *second minus first* in WER and MCD with a **paired** bootstrap 95% interval: both systems are scored on the same resampled sentences, which removes the sentence-difficulty noise. A negative Δ means the second system is better.

| Pair | Variable | Question |
|---|---|---|
| expA → expC, expB → expD | decoder objective | RQ1: flow matching vs regression |
| expA → expB, expC → expD | decoder depth | RQ2: 8 vs 4 blocks |
| subset → full, per experiment | training data | RQ3: 3,000 vs 10,480 utterances |

**How to read it:** if the interval contains 0, the difference is not reliable at this test-set size.

In [ ]:
# 1.4.1.5 Controlled comparisons with paired bootstrap CIs
pairs = [("objective", "full_expA", "full_expC"), ("objective", "full_expB", "full_expD"),
         ("depth", "full_expA", "full_expB"), ("depth", "full_expC", "full_expD")]
if "subset" in RUNS:
    pairs += [("data size", f"subset_{e}", f"full_{e}") for e in EV.EXPS]
rows = []
for var, a_, b_ in pairs:
    d = EV.paired_difference(per_utt, a_, b_)
    rows.append({"variable": var, "comparison": f"{a_} → {b_}",
                 "ΔWER % [95% CI]": f"{d['ΔWER_%']:+.2f} [{d['ΔWER_%_lo']:+.2f}, {d['ΔWER_%_hi']:+.2f}]",
                 "ΔMCD dB [95% CI]": f"{d['ΔMCD_dB']:+.2f} [{d['ΔMCD_dB_lo']:+.2f}, {d['ΔMCD_dB_hi']:+.2f}]",
                 "WER reliable?": "yes" if d["ΔWER_%_lo"] > 0 or d["ΔWER_%_hi"] < 0 else "no (CI contains 0)", **d})
comparisons = pd.DataFrame(rows)
comparisons.to_csv(EVAL_DIR / "comparisons.csv", index=False)
banner("1.4.1.5 Controlled comparisons (second minus first; negative = second is better)")
display(comparisons[["variable", "comparison", "ΔWER % [95% CI]", "ΔMCD dB [95% CI]", "WER reliable?"]].set_index("variable"))
print("  saved → results/eval/comparisons.csv")

### 1.4.1.6 Speed: RTF and latency
**What this cell does:**
1. Times end-to-end synthesis (text → wav) of the 12 prompts for each experiment: 2 warm-up runs (discarded), then 3 timed runs per prompt, i.e. **36 timed runs per experiment and device**. The GPU queue is synchronised before each clock reading.
2. Splits the time into front-end (normaliser + phonemiser), acoustic model and vocoder.
3. Repeats on the CPU, the realistic target for a deployment without a GPU.
4. Reports medians (robust to the occasional slow run) and the 90th-percentile latency; plots where the time goes.

**How to read it:** RTF < 1 means faster than real time. `RTF_acoustic` isolates the part that differs between experiments; the vocoder is identical in all four.

In [ ]:
# 1.4.1.6 Speed: RTF + latency, per stage, on the accelerator and on the CPU
devices = [DEVICE] + ([torch.device("cpu")] if DEVICE.type != "cpu" else [])
speed_raw = pd.concat([EV.measure_speed(PROMPTS, CKPT_ROOT, VOCODER_PATH, d) for d in devices], ignore_index=True)
speed = EV.summarise_speed(speed_raw)
speed_raw.to_csv(EVAL_DIR / "speed_raw.csv", index=False); speed.to_csv(EVAL_DIR / "speed.csv")
banner(f"1.4.1.6 Speed (medians of {int(speed.runs.iloc[0])} timed runs per experiment and device)")
display(speed.drop(columns="runs").round(4))
for d in speed.index.get_level_values(0).unique():
    s = speed.loc[d]
    kv(f"{d}: vocoder share of synthesis time", f"{(s.vocoder_s / (s.frontend_s + s.acoustic_s + s.vocoder_s)).mean():.0%}")
    kv(f"{d}: acoustic model, flow / regression", f"{s.acoustic_s[['expA', 'expB']].mean() / s.acoustic_s[['expC', 'expD']].mean():.1f}×",
       "10 decoder passes vs 1")

fig, ax = plt.subplots(1, len(devices), figsize=(6 * len(devices), 3.6), squeeze=False)
for a_, d in zip(ax[0], speed.index.get_level_values(0).unique()):
    s = speed.loc[d]
    bottom = np.zeros(len(s))
    for col, lab in [("frontend_s", "front-end"), ("acoustic_s", "acoustic model"), ("vocoder_s", "vocoder")]:
        a_.bar(s.index, s[col] * 1000, bottom=bottom, label=lab); bottom += s[col].values * 1000
    a_.set(title=f"Median synthesis time per prompt ({d})", ylabel="ms"); a_.legend(fontsize=8)
plt.tight_layout(); savefig("1_4_speed_breakdown"); plt.show()
print("  saved → results/eval/speed.csv, results/eval/speed_raw.csv")

### 1.4.1.7 Flow-matching step sweep
**What this cell does:** synthesises 50 test sentences with expA (flow matching, 8 blocks) at **1, 2, 4, 10 and 20 Euler steps** and reports WER, MCD and RTF for each. This is a quality/speed knob that only the flow decoder has. Cached in `results/eval/step_sweep.csv`.

**How to read it:** with 1 step the sampler barely leaves the starting point, so the output is close to the *mean* mel, like regression. If MCD falls as steps are removed, MCD is rewarding smoothness rather than detail, which is the caveat noted in §1.4.1.

In [ ]:
# 1.4.1.7 Flow-matching step sweep (expA), cached
sweep_path = EVAL_DIR / "step_sweep.csv"
if sweep_path.exists() and not SMOKE:
    sweep = pd.read_csv(sweep_path, index_col="steps")
else:
    sweep, sweep_raw = EV.step_sweep(TEST[:50], CKPT_ROOT / "expA" / "best.ckpt", DATA_DIR, VOCODER_PATH, DEVICE)
    sweep.to_csv(sweep_path); sweep_raw.to_csv(EVAL_DIR / "step_sweep_per_utterance.csv", index=False)
banner("1.4.1.7 expA: number of ODE steps at inference (50 test sentences)")
display(sweep.round(3))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for a_, col, lab in zip(ax, ["WER_%", "MCD_dB", "RTF"], ["WER (%)", "MCD (dB)", "RTF"]):
    a_.plot(sweep.index, sweep[col], "o-"); a_.set(xscale="log", xticks=list(sweep.index), xticklabels=list(sweep.index), xlabel="Euler steps", title=lab)
plt.tight_layout(); savefig("1_4_step_sweep"); plt.show()

### 1.4.1.8 Validation re-score: does the 300-clip monitor agree with the full validation split?
**What this cell does:** during training, `best.ckpt` was chosen on a fixed 300-clip validation monitor, to keep each epoch fast. This cell scores **every saved checkpoint** (epochs 75, 150, 225, 300 and `best`) on **all 1,310 validation clips** and on the monitor, for every run. It computes the mels of the validation clips not cached yet. Cached in `results/eval/val_rescore.csv`.

**How to read it:**
- the correlation between monitor and full-split losses: close to 1 means the monitor ranks checkpoints the same way;
- whether `best.ckpt` is also the lowest of the saved checkpoints on the full split.

In [ ]:
# 1.4.1.8 Re-score every saved checkpoint on all validation clips (cached)
rescore_path = EVAL_DIR / "val_rescore.csv"
if rescore_path.exists() and not SMOKE:
    rescore = pd.read_csv(rescore_path)
else:
    phon = json.loads((PROC_DIR / "phonemes.json").read_text())
    parts = []
    for run, root in RUNS.items():
        r = EV.rescore_validation(root, SPLIT["val"][:32] if SMOKE else SPLIT["val"], SUBSETS["val_monitor"], phon, MEL_DIR, DATA_DIR, EVAL_DIR, DEVICE)
        parts.append(r.assign(run=run))
    rescore = pd.concat(parts, ignore_index=True)
    rescore.to_csv(rescore_path, index=False)
banner(f"1.4.1.8 Validation loss on {32 if SMOKE else len(SPLIT['val']):,} validation clips vs the {len(SUBSETS['val_monitor'])}-clip monitor")
display(rescore.pivot_table(index=["run", "exp"], columns="checkpoint", values="all_val_total").round(4))
for (run, e), g in rescore.groupby(["run", "exp"]):
    r = np.corrcoef(g.monitor_total, g.all_val_total)[0, 1]
    best_is_min = g.loc[g.all_val_total.idxmin(), "checkpoint"] == "best" or g.all_val_total[g.checkpoint == "best"].iloc[0] <= g.all_val_total.min() + 1e-3
    kv(f"{run} {e}", f"r = {r:.3f}", "best.ckpt is also lowest on the full split" if best_is_min else "best.ckpt is NOT lowest on the full split")
print("  saved → results/eval/val_rescore.csv")

### 1.4.1.9 Interpretation
*Written after the final run; the numbers come from the tables above.*

`<to be completed with the final numbers: RQ1 objective, RQ2 depth, RQ3 data size, speed, step sweep, validation re-score>`

## 1.4.2 Qualitative evaluation (required)
Provide **≥10 text prompts** and corresponding **generated `.wav` outputs** (per your task requirements).
Include:
- The list of prompts you used
- Where the `.wav` files are saved (`wav_examples/…`)
- (Optional) a short comment on perceptual quality differences between experiments

> Spectrograms are optional unless explicitly required by your task sheet. If you include them, add 2–3 representative samples per experiment in `results/spectrograms/`.

**Our prompts:** the 12 prompts in §1.4.1.3, none taken from LJSpeech. They cover hard cases on purpose:

| # | Tests | # | Tests |
|---|---|---|---|
| 01 | everyday sentence (all letters) | 07 | technical vocabulary |
| 02 | greeting with a comma | 08 | tongue twister (repeated sounds) |
| 03 | **date, ordinal, money, time, abbreviation** | 09 | LJSpeech-style domain sentence |
| 04 | **question** intonation | 10 | **exclamation**, contraction |
| 05 | long sentence with clauses | 11 | plain instruction |
| 06 | **ambiguous abbreviations** (*Dr.*, *St.*), house number | 12 | **very long** sentence (25 words) |

**Where the files are:**
- `wav_examples/<exp>_<NN>.wav`: the four main experiments (48 files), prompts in `wav_examples/prompts.txt`;
- `results/wav_subset/<exp>_<NN>.wav`: the same prompts from the 3,000-utterance runs (RQ3);
- `results/eval/test_wavs/`: the first five test sentences from every system, next to the real recording.

### 1.4.2.1 Generate the prompt `.wav` files
**What this cell does:**
1. Writes `wav_examples/prompts.txt`.
2. Synthesises every prompt with every experiment's `best.ckpt` (and with the 3,000-utterance runs, if present). The noise seed is fixed per prompt, so the files are reproducible.
3. Shows audio players for four representative prompts (03 numbers, 04 question, 06 abbreviations, 12 long) for all four experiments, and the subset vs full-data versions of one prompt. All other files are on disk.

**Output to expect:** a file count per experiment, then the players.

In [ ]:
# 1.4.2.1 >= 10 prompts -> wav_examples/<exp>_<NN>.wav for every experiment
import tts.synth; importlib.reload(tts.synth)
from tts.synth import synthesize_to_wav
(WAV_OUT_DIR / "prompts.txt").write_text("".join(f"{i:02d}|{t}\n" for i, t in enumerate(PROMPTS, 1)))
outputs = {"full": WAV_OUT_DIR, "subset": RESULTS_DIR / "wav_subset"}
banner("1.4.2.1 Prompt synthesis")
for run, root in RUNS.items():
    for e in EV.EXPS:
        for i, t in enumerate(PROMPTS, 1):
            torch.manual_seed(i)
            synthesize_to_wav(t, exp=e, out_path=outputs[run] / f"{e}_{i:02d}.wav", ckpt_root=root, vocoder_path=VOCODER_PATH)
        kv(f"{run} {e}", f"{len(PROMPTS)} wavs", str(outputs[run].relative_to(PROJECT_ROOT)) + f"/{e}_NN.wav")

for i in (3, 4, 6, 12):
    print(f"\n  prompt {i:02d}: {PROMPTS[i - 1]}")
    for e in EV.EXPS:
        print(f"    {e} ({CONFIGS[e]['decoder']}, {CONFIGS[e]['dec_blocks']} blocks)")
        display(Audio(str(WAV_OUT_DIR / f"{e}_{i:02d}.wav")))
if "subset" in RUNS:
    print(f"\n  data size, expA, prompt 05: {PROMPTS[4]}")
    for run in ("subset", "full"):
        print(f"    {run} ({'3,000' if run == 'subset' else '10,480'} training utterances)")
        display(Audio(str(outputs[run] / "expA_05.wav")))

### 1.4.2.2 Spectrogram comparison
**What this cell does:** for three test sentences, plots the log-mel of the real recording, the vocoded recording and every experiment, on the same colour scale, and saves one PNG per sentence to `results/spectrograms/`. The first is shown here.

**What to look for:** horizontal **harmonic stripes** in voiced sounds (sharp in real speech; blurred if the decoder averages), **noise** at high frequencies in fricatives (*s*, *sh*), and the **timing** of words (gaps and lengths compared with the recording).

In [ ]:
# 1.4.2.2 Spectrograms: real vs vocoded vs every experiment, three test sentences
from tts.synth import load_tts, load_vocoder, synthesize
SPEC_DIR = RESULTS_DIR / "spectrograms"; SPEC_DIR.mkdir(parents=True, exist_ok=True)
voc = load_vocoder(VOCODER_PATH, DEVICE)
models = {f"{run}_{e}": load_tts(root / e / "best.ckpt", DEVICE) for run, root in RUNS.items() for e in EV.EXPS}
for k, (cid, text) in enumerate(TEST[:3]):
    gt = A.load_wav(DATA_DIR / "wavs" / f"{cid}.wav")
    wavs = {"real recording": gt,
            "vocoded real": voc(A.mel_spectrogram(gt)[None].to(DEVICE)).squeeze().detach().cpu().numpy()}
    for name, (m, ck) in models.items():
        torch.manual_seed(k)
        wavs[name] = synthesize(text, m, voc, ck["vocab"], ck["mel_stats"])[0]
    fig, ax = plt.subplots(len(wavs), 1, figsize=(11, 1.55 * len(wavs)), sharex=True)
    for a_, (name, w) in zip(ax, wavs.items()):
        a_.imshow(A.mel_spectrogram(w).numpy(), origin="lower", aspect="auto", vmin=-11.5, vmax=2.5, cmap="magma",
                  extent=[0, len(w) / A.SR, 0, 80])
        a_.set_ylabel(name.replace("_", "\n"), rotation=0, ha="right", va="center", fontsize=8); a_.set_yticks([])
    ax[-1].set_xlabel("seconds"); fig.suptitle(f"{cid}: {text[:90]}", fontsize=10)
    plt.tight_layout(); plt.savefig(SPEC_DIR / f"spec_{cid}.png", dpi=130, bbox_inches="tight")
    if k == 0:
        plt.show()
    plt.close(fig)
print("  saved →", ", ".join(str(p.relative_to(PROJECT_ROOT)) for p in sorted(SPEC_DIR.glob("spec_*.png"))))

### 1.4.2.3 Listening notes
*Filled in by the group after listening to `wav_examples/` (blind where possible: one person plays the files without naming the experiment).*

`<per experiment: naturalness, clarity, rhythm, artefacts; best and worst prompt, with file names; blind A/B preference counts expA vs expC>`

## 1.4.3 Coding practice & clarity
Briefly state:
- How you organised code (modules/scripts/configs)
- Any key engineering choices that improve reproducibility (fixed seeds, clear CLI args, etc.)

**Organisation:**
- **One package, written by this notebook.** `%%writefile` cells write every module to `tts/` (`text`, `audio`, `data`, `align`, `model`, `decoders`, `tts_model`, `train`, `synth`, `evaluate`), so the notebook, the scripts and the marker all run and read the same code.
- **Thin command-line scripts:** `scripts/synth.py` (text → wav) and `scripts/evaluate.py` (metrics); `python -m tts.train --config configs/expX.json` trains one experiment.
- **One config file per experiment** (`configs/expA–D.json`), generated from a shared base; `diff_configs()` (§1.3.3) prints the keys that differ, as evidence that each comparison changes one variable.
- **Shared print helpers** (`banner`, `kv`, `check`, `savefig`), so every cell's output has the same format; `check()` stops the notebook on a failed assertion.

**Reproducibility:**
- **Seeds everywhere:** split, subsets, data order, initialisation, validation noise and synthesis noise (per sentence / prompt).
- **Split fingerprint:** the MD5 of the ID lists is stored in every config and checkpoint.
- **Resumable, idempotent runs:** training resumes from `last.ckpt` and skips finished runs; the feature cache, the evaluation CSVs and the dataset download skip work already done and detect incomplete work.
- **No leakage:** vocabulary and mel statistics from the training split only; checkpoints selected on validation; metrics on the untouched test split.
- **Environment recorded** (§0.2, `results/environment.json`); a smoke mode (`A3_SMOKE=1`) runs the whole notebook end to end in minutes.

## 1.4.4 Attribution / external resources
List any:
- External libraries/models/toolkits used
- Pretrained checkpoints used
- Tutorials or code snippets adapted (with links in your report; in this notebook, name them clearly)

| Resource | Used for | Licence / source |
|---|---|---|
| **LJSpeech 1.1** (Ito & Johnson, 2017) | dataset | public domain, keithito.com |
| **HiFi-GAN V1, LJSpeech** (Kong et al., 2020) | pretrained vocoder (frozen); generator code copied to `tts/hifigan/` | MIT, via the Matcha-TTS repository (github.com/shivammehta25/Matcha-TTS) |
| **Matcha-TTS** (Mehta et al., 2024) | design of the flow-matching decoder (OT-CFM); our decoder is a simplified 1-D conv version | MIT |
| **Glow-TTS** (Kim et al., 2020) | Monotonic Alignment Search, re-implemented with Numba in `tts/align.py` | paper + MIT reference code |
| **keithito/tacotron** | abbreviation list in `tts/text.py` | MIT |
| **phonemizer + espeak-ng** (Bernard & Titeux, 2021) | grapheme-to-phoneme conversion | GPL-3.0 |
| **num2words** | spelling out numbers | LGPL |
| **Whisper small.en** (Radford et al., 2022), via Hugging Face `transformers` | automatic listener for WER/CER | MIT |
| **jiwer** | WER/CER computation | Apache-2.0 |
| **pyworld (WORLD), pysptk (SPTK)** | mel-cepstra for MCD (Kubichek, 1993) | MIT / BSD |
| **PyTorch, librosa, NumPy, pandas, matplotlib, soundfile, Numba** | general | BSD / ISC / MIT |
| Group prototypes: `notebook/collab_assignment_3.ipynb`, `notebook/En Li/*.ipynb`, `notebook/A3_TTS.ipynb` | starting point for the model code, the integrity checks and the environment report | own work |

Full references are in the report (Harvard style).

## 1.4.5 AI use statement (if any)
State whether you used generative AI tools, and for what (e.g., debugging, refactoring, documentation).
If used, specify what you verified yourself and what parts were AI-assisted.

`<group to complete: which tools were used and for what (e.g. code scaffolding, debugging, documentation); what the group verified itself (e.g. every number comes from this notebook's outputs; listening tests; citations checked against the papers)>`

# Final verification (before submission)

## Restart & run
Restart the kernel and run all cells you want the marker to execute, ensuring:
- No hidden dependencies on your local machine paths
- Commands/paths work from the submission root

## Confirm required artefacts exist
- `wav_examples/` contains generated `.wav` outputs (**≥10 prompts**)
- bjective metric results for each experiment are clearly reported in the notebook
- dependency file exists (`requirements.txt` / `environment.yml` / etc.)
- inference entrypoint works (single command that takes text and outputs wav)

## Marker instructions (one paragraph)
Write a short paragraph here telling the marker **exactly** how to:
1) install deps, 2) run training (or load checkpoints), 3) run inference demo, 4) locate results.

**Marker instructions.** (1) **Install:** `pip install -r requirements.txt`, plus the system package espeak-ng (macOS `brew install espeak-ng`; Ubuntu/Colab `apt install espeak-ng`). On Colab, cell 0.1 does all of this and downloads LJSpeech and the vocoder automatically. (2) **Training or checkpoints:** *Run all*. With the provided `checkpoints/runs_full/expA–D/` in place, training is skipped ("already trained for 300 epochs") and every later cell uses those checkpoints; without them, cell 1.3.4 trains all four (about 1–2 h each on a T4; set `RUN_EXPERIMENTS` in 0.1 to train a subset), or one experiment with `python -m tts.train --config configs/expA.json`. (3) **Inference:** `python scripts/synth.py --text "Hello world." --exp expA --out wav_examples/hello.wav --ckpt_root checkpoints/runs_full` (or `synthesize_to_wav()` in §1.3.6). (4) **Results:** metrics in §1.4.1 and `results/eval/` (`summary.csv`, `metrics.json`), audio in `wav_examples/`, figures in `results/figures/` and `results/spectrograms/`; `python scripts/evaluate.py` recomputes the test-set metrics.

### Final check: required artefacts
**What this cell does:** checks the artefacts the template lists and stops with a FAIL line if one is missing.

In [ ]:
# Final check: required artefacts
banner("Final verification")
n_wavs = {e: len(list(WAV_OUT_DIR.glob(f"{e}_*.wav"))) for e in EV.EXPS}
check(all(n >= 10 for n in n_wavs.values()), f"wav_examples/ has >= 10 prompts per experiment {n_wavs}")
check((EVAL_DIR / "metrics.json").exists() and set(main) <= set(summary.index), "objective metrics for every experiment (results/eval/metrics.json)")
check((PROJECT_ROOT / "requirements.txt").exists(), "dependency file requirements.txt")
check((PROJECT_ROOT / "scripts" / "synth.py").exists() and (DEMO_DIR / "cli_test.wav").exists(), "inference entry point scripts/synth.py ran (§1.3.6)")
check(all((CKPT_ROOT / e / "best.ckpt").exists() for e in EV.EXPS), "best.ckpt for expA–D")